# Kaggle job: seen-neutral

neutral

Inputs are resolved from the declared resource graph. No raw dataset re-upload is required.

# Prepare existing resources for the fixed multi-view evaluation

This reusable entry point executes one configured preparation operation. The launch kit binds existing Kaggle datasets and upstream outputs automatically; it does not re-upload the raw dataset. Models and the two selected scoring methods stay fixed. All scientific output, private checkpoints and reports remain separate.

Operations are explicit: method freeze, private RGB index, model-only RGB release, neutral reference geometry/correspondence, missing-feature and cache shards, cache merge, and post-prediction truth conversion. Only truth conversion opens anomaly outcomes, after complete model predictions verify. The trusted raw resource preparer may read mixed source metadata but exports only whitelisted neutral fields.

In [1]:
from pathlib import Path
import os
import json
RUNTIME_PROFILE = None
NOTEBOOK_RELATIVE = 'notebooks/input-preparation/current-kaggle-preparation.ipynb'
try:
    from pathlib import Path, PurePosixPath
    import hashlib
    import importlib.util
    import json
    import os
    import shutil
    import stat
    import sys
    import tempfile
    import zipfile

    # Bounded input lookup supports flat, namespaced and automatically expanded mounts.
    def _children(path):
        if not path.is_dir():
            return []
        children = sorted(p for p in path.iterdir() if p.is_dir())
        if len(children) > 250:
            raise ValueError('Source discovery bound exceeded; set VPI_CODE_ROOT explicitly')
        return children

    def _source_candidates():
        explicit = os.getenv('VPI_CODE_ROOT')
        if explicit:
            return [Path(explicit).expanduser()]
        mount = Path('/kaggle/input')
        roots = [Path.cwd(), *Path.cwd().parents]
        roots += [p for p in _children(mount) if p.name not in {'datasets', 'notebooks', 'models'}]
        for category, depth in (('datasets', 2), ('notebooks', 2), ('models', 4)):
            current = [mount / category]
            for _ in range(depth):
                current = [child for parent in current for child in _children(parent)]
                if len(current) > 1000:
                    raise ValueError('Source mount inventory exceeded its bound')
            for parent in current:
                roots += _children(parent / 'versions') if (parent / 'versions').is_dir() else [parent]
        return [root / suffix for root in roots for suffix in ('', 'code', 'repository-files', 'source')]

    def _archive_source():
        configured = os.getenv('VPI_SOURCE_ARCHIVE')
        if not configured:
            raise FileNotFoundError('Mount the prepared source package or set VPI_CODE_ROOT explicitly')
        archive = Path(configured)
        expected = os.getenv('VPI_SOURCE_ARCHIVE_SHA256')
        if not expected or archive.stat().st_size > 256 * 1024 * 1024 or hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise ValueError('Explicit source archive checksum or size check failed')
        destination = Path(os.getenv('VPI_SOURCE_DEST', '/kaggle/working/current-source')).resolve()
        if Path('/kaggle').is_dir() and not destination.is_relative_to(Path('/kaggle/working')):
            raise ValueError('Extracted source must be under /kaggle/working')
        if destination.exists():
            raise ValueError('Source destination exists; verify it with VPI_CODE_ROOT instead of overwriting')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = Path(tempfile.mkdtemp(prefix='current-source.partial-', dir=destination.parent))
        try:
            with zipfile.ZipFile(archive) as bundle:
                infos = bundle.infolist()
                names = [i.filename for i in infos]
                if len(names) != len(set(names)) or sum(i.file_size for i in infos) > 256 * 1024 * 1024 or bundle.testzip():
                    raise ValueError('Source archive inventory, size or CRC is invalid')
                for info in infos:
                    name = info.filename
                    path = PurePosixPath(name)
                    if info.is_dir() or path.is_absolute() or '\\' in name or any(p in {'', '.', '..'} for p in name.split('/')) or stat.S_ISLNK(info.external_attr >> 16):
                        raise ValueError('Unsafe source archive member')
                    output = temporary / name
                    output.parent.mkdir(parents=True, exist_ok=True)
                    output.write_bytes(bundle.read(info))
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary)
            raise
        return destination

    _matches = sorted({p.resolve() for p in _source_candidates() if (p / 'prepared_source_manifest.json').is_file()})
    if len(_matches) > 1:
        raise ValueError('Multiple prepared source roots found; set VPI_CODE_ROOT explicitly')
    CODE_ROOT = _matches[0] if _matches else _archive_source()
    _manifest = json.loads((CODE_ROOT / 'prepared_source_manifest.json').read_text())
    # Verify the bootstrap itself before executing any code from the mounted package.
    _helper = CODE_ROOT / 'scripts/current_notebook_runtime.py'
    if _helper.is_symlink() or hashlib.sha256(_helper.read_bytes()).hexdigest() != _manifest.get('files', {}).get('scripts/current_notebook_runtime.py'):
        raise ValueError('Prepared bootstrap source checksum differs')
    _spec = importlib.util.spec_from_file_location('current_notebook_runtime', _helper)
    _runtime = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_runtime)
    _launch_helper = CODE_ROOT / 'scripts/current_launch.py'
    if hashlib.sha256(_launch_helper.read_bytes()).hexdigest() != _manifest['files']['scripts/current_launch.py']:
        raise ValueError('Launch helper checksum differs')
    sys.path.insert(0,str(CODE_ROOT/'scripts'))
    from current_launch import configure_runtime, resolve_job
    LAUNCH_PLAN = json.loads('{"continuation":{"added_adapter_files":["scripts/current_official_region_ids.py"],"changed_transport_files":["notebooks/input-preparation/current-native-region-inference.ipynb","scripts/build_native_regions_notebook.py","scripts/current_kaggle_queue.py"],"external_completed_job_count":68,"new_full_plan_sha256":"0e9be1f93ade48e22276df3f0f54d4010b02c0ef40e907235be463480658e5f7","new_source_snapshot_sha256":"d2c37bb6d8824d2e085c6206b70e5ddfb9927eef25336f58bcd41f33bc1a7296","old_full_plan_sha256":"0e9be1f93ade48e22276df3f0f54d4010b02c0ef40e907235be463480658e5f7","old_source_snapshot_sha256":"feb25c1bd8f64fe43e6eb9701b21ed9b10c45a86e27d103106bf65a41d446c66","pending_jobs":{"seen-regions-00":"failed","seen-regions-01":"failed","unseen-rgb":"completed_without_required_receipt"},"schema":"vpi-official-region-id-transport-retry/v1","scientific_contract_changed":false},"core_views":[2,3,5],"diagnostic_views":[1],"external_completed_jobs":{"08f1":{"completion_sha256":"66cca80043c1e496632b92f79cb882ad378804707dacead6cee46312a674e1a8","kernel_ref":"esreekarreddy/vpi-current-08f1","outputs":{"08f1":"01351302ce18183ad561e78e1464744d3ae96375e1080f9d5cffb0115b4d360e"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"09a":{"completion_sha256":"4eb50134be185ca8752624625f424491cd1f63417b5c43700d4440c52d85dfff","kernel_ref":"esreekarreddy/vpi-current-09a","outputs":{"09a":"91d73ad63ae96f7219b35488b03e28af570b592fdbe3d70cfc8022982236968f"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"09b":{"completion_sha256":"ee612f352a68354defa3f0c0f785565f4906bca7cd0f34870422d2d3e92dffa4","kernel_ref":"esreekarreddy/vpi-current-09b","outputs":{"09b":"087188994d32f0ef8181a70cc5ebfbf2ddf8ebcbb34274eb57b66d7069a89526"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"09b1":{"completion_sha256":"095c24fdc8f257b92ba10148e8a98206af6d5f0d8b1c2ebab6b866688d495105","kernel_ref":"esreekarreddy/vpi-current-09b1","outputs":{"09b1":"a4b56aa0556ca0e30dfbdc4c89cc752677dac194e47abb0de52e1d439a039221"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"09c":{"completion_sha256":"4b15e9efc7198a96dc1e19599030fb63d2320525820166c688a37f0804c287ba","kernel_ref":"esreekarreddy/vpi-current-09c","outputs":{"09c":"c6c31649eda7dad81038d9e18cb0a2026738abf0794b79eb97a4127a0b0f0a0e"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"09d":{"completion_sha256":"baf20ae584bc8726a046fc7aac404fb3916b7646e5cb8dd1fc6499db4690de9e","kernel_ref":"esreekarreddy/vpi-current-09d","outputs":{"09d":"351bbaa12337ae120a425c72bcc534387a1f523e845cda40f7c9b956617aa780"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"calibration-cache":{"completion_sha256":"ca948e671cfdb8448a53e9f832233c5175e30aec0757fc267fa15c41117f595e","kernel_ref":"esreekarreddy/vpi-current-calibration-cache","outputs":{"calibration-cache":"01351302ce18183ad561e78e1464744d3ae96375e1080f9d5cffb0115b4d360e"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"calibration-cache-00":{"completion_sha256":"a426bb08891adb725754a26aba48971e73726e5cd6dcd1025c6920262a1c53e1","kernel_ref":"esreekarreddy/vpi-current-calibration-cache-00","outputs":{"calibration-cache-00":"1a1e041982db1be4334371777737b3b675d8d6e3a6c48d1206b84839e76ec9cc"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"calibration-cache-01":{"completion_sha256":"e607ff9b0e20303cccc8513c3adb6fe68218d21dcd04e2e452982cc756a3c7a2","kernel_ref":"esreekarreddy/vpi-current-calibration-cache-01","outputs":{"calibration-cache-01":"a16c4777509b2b317abcfa3064b2bd0fb30c7bcba7fd53055113b2b48ff68972"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"calibration-cache-02":{"completion_sha256":"a1359a03fa5049fcb388fc60190df620027231861a187e064d7b3e7c47ba88a1","kernel_ref":"esreekarreddy/vpi-current-calibration-cache-02","outputs":{"calibration-cache-02":"011238b5fde2410612bd377da8107500d93f64362b990faf10af907f0c7a5c6f"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"calibration-cache-03":{"completion_sha256":"225c929cb127a464b701fe2dae8930459e2b27dbdf1afc7a896bd8c339d561fb","kernel_ref":"esreekarreddy/vpi-current-calibration-cache-03","outputs":{"calibration-cache-03":"a5107cbacc600d9dc4b410c3fa8cdefa70e789c3907816a686c72adc643647c8"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"calibration-cache-04":{"completion_sha256":"e7ccde4d369bbf91e290479ff1a504d10bb5b8bdcbe172facda474ab709e172f","kernel_ref":"esreekarreddy/vpi-current-calibration-cache-04","outputs":{"calibration-cache-04":"8c2824c9a719b5da17080900ba63d7ae3aaed8f88c35ab50307a6cca22994e2f"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"calibration-cache-05":{"completion_sha256":"478bfb3f22eae2e70b88136b922e284616ee1e1f199e4523aaa6f3f4f846e336","kernel_ref":"esreekarreddy/vpi-current-calibration-cache-05","outputs":{"calibration-cache-05":"cc3e03f48dd4f5e164e967dde4b3e16ee524e83966893c675cc657918569de04"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"calibration-geometry":{"completion_sha256":"a69919b5fb186620af34109a3a7336af4b9aa4df1ff734e4de3f592bdc71e2ab","kernel_ref":"esreekarreddy/vpi-current-calibration-geometry","outputs":{"calibration-geometry":"b87d30b20913c25deb0974feec743540a8927381aa4a82583baac3501bea86b3"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"calibration-index":{"completion_sha256":"bcbcd6b7a14648f87c69aa1aecc291567081a33cc221ed0b6a940aea73e6a20e","kernel_ref":"esreekarreddy/vpi-current-calibration-index","outputs":{"calibration-index":"aa7e4a86350b100e0507eafc8aacec71286e7100117781f9a72ce9a3cf2fb5ea"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"calibration-neutral":{"completion_sha256":"95381dcc66b5422a9d5c41a9fc157b5f9fa7100c591e7a3a6a3bcd73d89ba0c5","kernel_ref":"esreekarreddy/vpi-current-calibration-neutral","outputs":{"calibration-neutral":"be807c28bef8d685ece1c226a5b93f8734ee51f3e6798c1f39c48b376dd4960b"},"plan_sha256":"0c2334bdc964543b5d68ff56b0505f7bbe8be05fc866399cdcbf180b017d6821","source_snapshot_sha256":"0eea1717ea8e87bf814d32ddbed0b9e03fb6d56b55c7d51d141588ab506beb31"},"calibration-regions":{"completion_sha256":"a5be1430a40c193db139b91bdeeb7b5b623ebd247f269b724bf0500e79487da5","kernel_ref":"esreekarreddy/vpi-current-calibration-regions","outputs":{"calibration-regions":"0830ed1664dbaea8dbc47e60704c6df2ff02b81af49fc01c09de8b912f1a61c0"},"plan_sha256":"0c2334bdc964543b5d68ff56b0505f7bbe8be05fc866399cdcbf180b017d6821","source_snapshot_sha256":"0eea1717ea8e87bf814d32ddbed0b9e03fb6d56b55c7d51d141588ab506beb31"},"calibration-regions-00":{"completion_sha256":"e9a3978d0cc8f0bae9e5fcbb091e428da9a5feaecb9d017e3330bb273e964d80","kernel_ref":"esreekarreddy/vpi-current-calibration-regions-00","outputs":{"calibration-regions-00":"9ba7ad6270222d09b749e9220b38cc7b12d9bee57a0b5161875840b978f3b6d7"},"plan_sha256":"0c2334bdc964543b5d68ff56b0505f7bbe8be05fc866399cdcbf180b017d6821","source_snapshot_sha256":"0eea1717ea8e87bf814d32ddbed0b9e03fb6d56b55c7d51d141588ab506beb31"},"calibration-regions-01":{"completion_sha256":"902f440dd676dba66e2965cd486ae9977b562dddc5bc11804c93d6f54b46193a","kernel_ref":"esreekarreddy/vpi-current-calibration-regions-01","outputs":{"calibration-regions-01":"307354cc059c68f2762a00485cb7b8133b04cda1d08c9ed24a96a0c8571c9343"},"plan_sha256":"0c2334bdc964543b5d68ff56b0505f7bbe8be05fc866399cdcbf180b017d6821","source_snapshot_sha256":"0eea1717ea8e87bf814d32ddbed0b9e03fb6d56b55c7d51d141588ab506beb31"},"calibration-regions-02":{"completion_sha256":"5754a8d6b02043d7aabcfc6c2fe24063526fe45ef5846815d30c6ce41914034e","kernel_ref":"esreekarreddy/vpi-current-calibration-regions-02","outputs":{"calibration-regions-02":"b37f7763c0df892127e842531902b4899a5024f8c02457e1f94952c3b4cec302"},"plan_sha256":"0c2334bdc964543b5d68ff56b0505f7bbe8be05fc866399cdcbf180b017d6821","source_snapshot_sha256":"0eea1717ea8e87bf814d32ddbed0b9e03fb6d56b55c7d51d141588ab506beb31"},"calibration-regions-03":{"completion_sha256":"53a52f40375bb57b5168a76cf20e93831b0d539c795aab76437bcf983eac9b99","kernel_ref":"esreekarreddy/vpi-current-calibration-regions-03","outputs":{"calibration-regions-03":"4c4644fef1eabeb6d24c62ae449eb22041b069e071ae1fed7b55224f45560766"},"plan_sha256":"0c2334bdc964543b5d68ff56b0505f7bbe8be05fc866399cdcbf180b017d6821","source_snapshot_sha256":"0eea1717ea8e87bf814d32ddbed0b9e03fb6d56b55c7d51d141588ab506beb31"},"calibration-regions-04":{"completion_sha256":"cc720344d8de0d5ddb389546812775d4c2738e127d330cd27bc1f6b05b284a9d","kernel_ref":"esreekarreddy/vpi-current-calibration-regions-04","outputs":{"calibration-regions-04":"33979168a4e8f6c9322518dbbb55f8e52243fa7f8466bc06104446ad714c8337"},"plan_sha256":"0c2334bdc964543b5d68ff56b0505f7bbe8be05fc866399cdcbf180b017d6821","source_snapshot_sha256":"0eea1717ea8e87bf814d32ddbed0b9e03fb6d56b55c7d51d141588ab506beb31"},"calibration-regions-05":{"completion_sha256":"047e19697e47ae1632c531e7c53ace59c20ad28aa7a57bc1cdf305a9f70e047f","kernel_ref":"esreekarreddy/vpi-current-calibration-regions-05","outputs":{"calibration-regions-05":"af6c2c218a6d84fe7983aa2957979f5d4689d7c7f7e16dd1aaf8d61e181e6454"},"plan_sha256":"0c2334bdc964543b5d68ff56b0505f7bbe8be05fc866399cdcbf180b017d6821","source_snapshot_sha256":"0eea1717ea8e87bf814d32ddbed0b9e03fb6d56b55c7d51d141588ab506beb31"},"calibration-rgb":{"completion_sha256":"8408b1a5ddcd63240d08dce032e6572a79c1b1d436fa9506414e7133961141e1","kernel_ref":"esreekarreddy/vpi-current-calibration-rgb","outputs":{"calibration-rgb":"542ade928cbc87b0ecfd902616c39af96c58748b527064f3d592fa9691a2bb12"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"calibration-truth":{"completion_sha256":"1c50bbbbf800b9d1c07360b6501105b7d23d448ded4f6604d9b45ba83d8f54b6","kernel_ref":"esreekarreddy/vpi-current-calibration-truth","outputs":{"calibration-truth":"c3a18e4c4691359fbb7946ab01dda333f5d49a3dcf84903dd5e11c2135e9b97a"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"development-cache":{"completion_sha256":"744dd3fb8e238ea48b6e2a6674a38f135c06a637a667e022ed74afbae7df67a9","kernel_ref":"esreekarreddy/vpi-current-development-cache","outputs":{"development-cache":"b8548176bc1cf0608b59157fb1eb2e00ce2acfdea50d9ef6bab288e624d04816"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-00":{"completion_sha256":"cbffd500e9f4614bb4a9efb0aabab084d17de2510af45ef48ddcada7ecee8f80","kernel_ref":"esreekarreddy/vpi-current-development-cache-00","outputs":{"development-cache-00":"391764e98ed6bd6ef1d9b8d0ef38692fc4f5cce8057bc10e38b92c9f870f93ba"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-01":{"completion_sha256":"0711188977a95e31dc50b487177d6cdf4dacddfee224291ee39ca030c0e225a9","kernel_ref":"esreekarreddy/vpi-current-development-cache-01","outputs":{"development-cache-01":"023091ed2dd8fdc7107e92a675fd802438d41ea7ee5cb2594a25de6b510220d0"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-02":{"completion_sha256":"04407768ba58c7fdd16bba2845bcd9f2f7b069d9fdc1dda90d0de2e88c861a7c","kernel_ref":"esreekarreddy/vpi-current-development-cache-02","outputs":{"development-cache-02":"8ccd0b88d09a6afc0c11d6a29f8983d62c59e719a10b920bdd2b1412f39441ce"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-03":{"completion_sha256":"c16830b74796ebb8551e939b6854e6b910e2a89b1d57b8b5cfe398623df89cdf","kernel_ref":"esreekarreddy/vpi-current-development-cache-03","outputs":{"development-cache-03":"60ddd249f26849dee72d67a6c93ebbc2af0ea82d21156501c9f035698d7e9c16"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-04":{"completion_sha256":"2f3eae26e4f1019244ee355e166ec8dd7deeae80832a51fbb669de6418f41ec1","kernel_ref":"esreekarreddy/vpi-current-development-cache-04","outputs":{"development-cache-04":"2005451cee34715d3cf50fa41c4ddf08b2de0e6fe4206c261b94b8bab20283e0"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-05":{"completion_sha256":"810f641238ca3ca306affc8fdeea72b9572197172068028510f39a1c7e97bf9a","kernel_ref":"esreekarreddy/vpi-current-development-cache-05","outputs":{"development-cache-05":"1cb58e46ef3afef0fda3476719089cee30235dde0bb403fc822ef4dcd57ce40b"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-06":{"completion_sha256":"bc9f89bda3448225e2e5a2dccfb4ed7bb35ee0a91ab15de8ec2c4ef3f54d773c","kernel_ref":"esreekarreddy/vpi-current-development-cache-06","outputs":{"development-cache-06":"931162fbfc5d020089ba7eaa5cd0f4b47e6ebe7579b56e15e52c492b80af1af0"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-07":{"completion_sha256":"ba2ccb83946e0c85e2d41ad00a57a872a5c0b4f0820e7b0adc43eab0ae83b138","kernel_ref":"esreekarreddy/vpi-current-development-cache-07","outputs":{"development-cache-07":"32728878cae2b833753fbdf624900d5c7e8fba6b47cb85d58670e82a9403d757"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-08":{"completion_sha256":"192a239e02224c4f26cb545d9b40358ab972a601cdef0a1b1312c42cae9ade23","kernel_ref":"esreekarreddy/vpi-current-development-cache-08","outputs":{"development-cache-08":"78c29d6d344ee0809915ddbf1f6bf2a94984d3965ac4c4b57012ca36c6d68dc6"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-cache-09":{"completion_sha256":"f1e514a515988492d8f35d5cccf086ef1d8a90b97ad8fec3d44b1b3210534c8b","kernel_ref":"esreekarreddy/vpi-current-development-cache-09","outputs":{"development-cache-09":"87a43a15a5d37e353826d9896c5a4fac49c6554ad03e055e658cc5b317808973"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-evaluate":{"completion_sha256":"38f62d9118f5ea0637fad88c9b399eaf1ddfab29ccd164daeed4cf62b10c49ac","kernel_ref":"esreekarreddy/vpi-current-development-evaluate","outputs":{"development-evaluate":"e6c80e78c86957d0ba1dfb486dc0ad251178f728233003c2cf270eec90f31652"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-geometry":{"completion_sha256":"dd0fec22d009ddf290e3aa4e9f5bfb753462acdbf14c6f585ab263c04b11203a","kernel_ref":"esreekarreddy/vpi-current-development-geometry","outputs":{"development-geometry":"c9b8abb9b4f6593e6435e19ba00983c327f86e50098dc5b475b61829f329e895"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-index":{"completion_sha256":"a9d780cb7de177080f2ad3bda21514094c6f0552ba1dcffbd15985db03fa478b","kernel_ref":"esreekarreddy/vpi-current-development-index","outputs":{"development-index":"a813ed0247bf035533348930c450c850e274da96c4d3506da875dfbdee506736"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-neutral":{"completion_sha256":"c25f4b771e178bc2850b3502f490534be2f9c954a0f7acbdbb08c52b327a7af4","kernel_ref":"esreekarreddy/vpi-current-development-neutral","outputs":{"development-neutral":"8d8bf36f6e8163778a8203850e12e973f9f2aeba48d9c5e660aaec7aea7beb9d"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-predict":{"completion_sha256":"0451012af6cf9bb5c53e512cc225ec23c5209802c99dcc3c8940c263660bd59a","kernel_ref":"esreekarreddy/vpi-current-development-predict","outputs":{"development-predict":"e372c4c80bf485f2d132f73db584fb8afd028859038b3e7b7c950c8969d1ba9d"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"development-truth":{"completion_sha256":"d1e0eb48c34d5c57eb67ea36eda73cf9465f103194f703535f70d1f315094cb6","kernel_ref":"esreekarreddy/vpi-current-development-truth","outputs":{"development-truth":"7e3dfe798160ad6d571573792fc19b8657bdd307e257a2d74d4b3e862ec0924f"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"freeze":{"completion_sha256":"9d8d9ac75a8e957dd0dfdccab412d2de3e225d8f41d1f8fbde6405bcf11933d2","kernel_ref":"esreekarreddy/vpi-current-freeze","outputs":{"freeze":"57b8a32532c249f462f77240e277e9c35a206b7d18c34f63a410a5d280f71b50"},"plan_sha256":"c8434dd5bc2c04b02a2bfd2ad4055ede764fd4aa551e26109478557b708c0f7f","source_snapshot_sha256":"6c022aa28cbee9fa48931b3d8d3de0a7e8a92fb9ce7685fe9f17521beeb78e2c"},"internal_lock-cache":{"completion_sha256":"b622f1f5f7a409bc53ca49c30232c653f70e6a8ad2671deefb8e47c3538fda52","kernel_ref":"esreekarreddy/vpi-current-internal-lock-cache","outputs":{"internal_lock-cache":"a4b56aa0556ca0e30dfbdc4c89cc752677dac194e47abb0de52e1d439a039221"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-cache-00":{"completion_sha256":"eed9d4499485488da04b40346b9809d102454c8ad9ab974b4bf7f3a4cddfbc4d","kernel_ref":"esreekarreddy/vpi-current-internal-lock-cache-00","outputs":{"internal_lock-cache-00":"0234bfe6fc453be91ac0208c5795093244c765ff056935e65f3aa816a486b65f"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-cache-01":{"completion_sha256":"1a44569565fb2d465d60409b7beeeb8712c1705746725ff3f6019a006e719f0e","kernel_ref":"esreekarreddy/vpi-current-internal-lock-cache-01","outputs":{"internal_lock-cache-01":"e851d23b547cc55801e4702844141ccf393c3da2f2bb70a93282746f528098cd"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-cache-02":{"completion_sha256":"921d857f21136efd3e3ab74ac54d4e86827237ac173bbd345f47045250956ecc","kernel_ref":"esreekarreddy/vpi-current-internal-lock-cache-02","outputs":{"internal_lock-cache-02":"86a3382d1f36f340485a6c2d06f1f48e29cece7390b737769b31c48de800616f"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-cache-03":{"completion_sha256":"abe61143a16c5d102df6449bb035219c38596f719377238a031a171d8241918c","kernel_ref":"esreekarreddy/vpi-current-internal-lock-cache-03","outputs":{"internal_lock-cache-03":"e8bf050e46a73d5a95a54e4870bd19cee175b6e472b55c3e6037544facaf9482"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-geometry":{"completion_sha256":"f4db683d86b906542eb30fe83412744dd4065408dd932d74ddfc732b1f7fcb25","kernel_ref":"esreekarreddy/vpi-current-internal-lock-geometry","outputs":{"internal_lock-geometry":"d7d59f794aafa26b9d8e2f0e2935ad1129887010ec1ee9c718cbe347425a4fb5"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-index":{"completion_sha256":"23d5383740092f7a7140875f14f1594de46f8aecd3a35131b9edf602f570179d","kernel_ref":"esreekarreddy/vpi-current-internal-lock-index","outputs":{"internal_lock-index":"901263a0104f4d753a78e3a2d4cacdcc6015bf3a63fadb2ff5aeeb227e203726"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-neutral":{"completion_sha256":"f380c6055b516c5982cddf34f2a2dee714fd6cd0abc0c173fc04b939ddd3eebe","kernel_ref":"esreekarreddy/vpi-current-internal-lock-neutral","outputs":{"internal_lock-neutral":"4fa4aec3afa6e146bcf85ee6010a16a9ca85f7598dc94d5571b78583da85512b"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-regions":{"completion_sha256":"396985291fe5248113ae05ffaa2380bb07920238e5d89014f188ddd1ed19b505","kernel_ref":"esreekarreddy/vpi-current-internal-lock-regions","outputs":{"internal_lock-regions":"289bd402472e56e7417c57b8f0084c7feee78378b8b4123a915ecd06c7a32da5"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-regions-00":{"completion_sha256":"2b5c3bcf929a20f7d9c505f05407c6943bfe849f94d72ff4c398b95f1cf20350","kernel_ref":"esreekarreddy/vpi-current-internal-lock-regions-00","outputs":{"internal_lock-regions-00":"30e338fe7e91d73fd72deeae909b0be6e6a908eb0015da52f807b283f8d95e76"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-regions-01":{"completion_sha256":"b9ed797afd0106612070e288be2ce81c759270d819d1c7caff29a6ca410913e8","kernel_ref":"esreekarreddy/vpi-current-internal-lock-regions-01","outputs":{"internal_lock-regions-01":"3f1d06f7f75c9fe41df64b82af650b42ccbbbc61d5e2f0c4f815f007de318d2a"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-regions-02":{"completion_sha256":"373e213a431d30f8125ec3711b6d22c7349bfd5fcbc054cb9308bc66100d873e","kernel_ref":"esreekarreddy/vpi-current-internal-lock-regions-02","outputs":{"internal_lock-regions-02":"5691d1e90d8ea8d0726cb4a456bf417d1ebbf86d1600f26b611564cf4aea5b51"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-regions-03":{"completion_sha256":"0703e6a16a4245049cae8c78f15db91f56c52c285649775a981deddb14edfa16","kernel_ref":"esreekarreddy/vpi-current-internal-lock-regions-03","outputs":{"internal_lock-regions-03":"a441e15b4f19db7fa48eb2d11543ee075e1a2c28370b27558696c79f6261b7af"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-rgb":{"completion_sha256":"5f175062d93d0f8ba17f813c296f27add18fb4b26d06a7f082b9265cc62a8ff6","kernel_ref":"esreekarreddy/vpi-current-internal-lock-rgb","outputs":{"internal_lock-rgb":"07063d5ce596a035f5c693f35b4d7e6136ac041265c587bcc6f94775d7e26094"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"internal_lock-truth":{"completion_sha256":"894a739349fe439739262de20f2ed7ff33bb96544f227dd4c892187c16fc6e90","kernel_ref":"esreekarreddy/vpi-current-internal-lock-truth","outputs":{"internal_lock-truth":"907f3a899a5c59a3c98d4e3a0474c49bb15dcff3c0bec3d1af2684ed162753db"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"official-exposure":{"completion_sha256":"0f9c10cf0ad63c7e3b290b7d42fe61a70189358c07a9fec7a77daafcf7238bf2","kernel_ref":"esreekarreddy/vpi-current-official-exposure","outputs":{"official-exposure":"041a3c3d2b77bf2503172ac3436f36b05a707265218129136bb7f937760ed8fe"},"plan_sha256":"5be012a5876b65383e7bf8e65ef27e0ae43887c014cb07be1071484d9e9fb406","source_snapshot_sha256":"feb25c1bd8f64fe43e6eb9701b21ed9b10c45a86e27d103106bf65a41d446c66"},"seen-geometry":{"completion_sha256":"2b52461fa9fe7e6a4d897624238ae4135c295003f09daeea7f30906b456e662a","kernel_ref":"esreekarreddy/vpi-current-seen-geometry","outputs":{"seen-geometry":"52ab97d718ce199bfb73503cb17a7dd503d1bbd8228587bfac5a90589294492e"},"plan_sha256":"1959511984e0d05409949d738a60358bc3351f64c7e0f466731fe3d97275d1f1","source_snapshot_sha256":"aba52cc54562db97652ca69f6ed6bef72b82846ce661d2d12fa54b919c1bbda2"},"seen-index":{"completion_sha256":"c1859939b77586bf4238e444b4a202f2f87942b414f27a25f5c44b339aa791c3","kernel_ref":"esreekarreddy/vpi-current-seen-index","outputs":{"seen-index":"30e8a04d606bf0f423635fff06ace30e2f1e8b345b4300908af7326e4a371506"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"},"seen-rgb":{"completion_sha256":"ce1b5310ec1f0f7f4cbabb896036cfdce3effe8ca979a47aa27c4c8c75e96e00","kernel_ref":"esreekarreddy/vpi-current-seen-rgb","outputs":{"seen-rgb":"2d641015f5c612d585b17514ed2106cb48bc4fd2d2b2916bb26ff101dba8ab53"},"plan_sha256":"5be012a5876b65383e7bf8e65ef27e0ae43887c014cb07be1071484d9e9fb406","source_snapshot_sha256":"feb25c1bd8f64fe43e6eb9701b21ed9b10c45a86e27d103106bf65a41d446c66"},"setup-cpu":{"completion_sha256":"85054d08f8a755d6e10ba24f612a0f293ed959cdd5b9279beb29cfb32fad07ab","kernel_ref":"esreekarreddy/vpi-current-setup-cpu","outputs":{"runtime-cpu":"246cbde33f8c880f6a26bb23dad896b44ec9b39234e7dcd439df5c22c4aebd5d","wheels-cpu":"ab746146158ca180f5838e28f62dc19fb291df241dd13b1e12ee558c739e3332"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"setup-dino":{"completion_sha256":"ab46de5232c36f57ef98451163bb6afb9f18311d4b84aaa72cd9942f7b56a0cc","kernel_ref":"esreekarreddy/vpi-current-setup-dino","outputs":{"models":"f407fdc4c45291526d82bf39fa0f5a841a6dbd301a6d6febace9d3eaa2f79698","runtime-dino":"4462eef3a141c0d647b0d418c91e98e134522240f678027d7bef8e0aebc15a0f","wheels-dino":"908cd70d0da8788f6cfd2e752d7dc1ddef11c5d81349653eaa08546b3e014207"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"setup-regions":{"completion_sha256":"cfd42af510a6bc68b6fe49291fcef830838534b1f618576abe9bf79d388b6cb8","kernel_ref":"esreekarreddy/vpi-current-setup-regions","outputs":{"runtime-regions":"f92df2f676fda1c06794c22ea04e996ce298958f7af082ae64c250052c4a4d2c","wheels-regions":"b95309ec18a1c4de59cdb09fa1d7817668db2d7a1432fe8254f86a378da0893f"},"plan_sha256":"7afdfc56c1573f7333d6efbd14145cd1a1b41210ec13ab0ec15b26c10cad23eb","source_snapshot_sha256":"eaed514f5f76c57e886e0568d2c2a352ba174bde4b617767f95ffdd90b2aa006"},"setup-tracking":{"completion_sha256":"1f0c2bd59e7f8ea809fee95239bb0d58165850de7e66a3c8e12a3641cfda55f9","kernel_ref":"esreekarreddy/vpi-current-setup-tracking","outputs":{"runtime-tracking":"fba340a3438b742d110fd053cd5748ef2d6715d965ae70b50330b600b901192c","wheels-tracking":"00d27db6c0b0a4e826e43d8a353274c81b062d6bbefaa3f5d0d2d6b0a42d12ae"},"plan_sha256":"d5f7e04f1a5e9e35a618eada3486d5ef242270568938406dbdf53bb5b04b4c49","source_snapshot_sha256":"ebc3467dd55a33467970bdee07ad0e1865de69845fe061f4437a6f651012c2bf"},"unseen-geometry":{"completion_sha256":"5393d0aa99fcbf7bc74c9b9adfc7e0701ac2093bf492ce47e77dd291069d2962","kernel_ref":"esreekarreddy/vpi-current-unseen-geometry","outputs":{"unseen-geometry":"4815f8dabb475574d85bc587571013e48592f4f0d96ffbc911ac8d95ca277932"},"plan_sha256":"1959511984e0d05409949d738a60358bc3351f64c7e0f466731fe3d97275d1f1","source_snapshot_sha256":"aba52cc54562db97652ca69f6ed6bef72b82846ce661d2d12fa54b919c1bbda2"},"unseen-index":{"completion_sha256":"b5177a737c11317aaa1a12a34c53e06d31156308d78cedc2117d687a88db414b","kernel_ref":"esreekarreddy/vpi-current-unseen-index","outputs":{"unseen-index":"37ea69cd586e234b05e6d5d9cb0ad6941013e6b1a972de70df1e81ecfd15a392"},"plan_sha256":"aa32b2e29836bfd5bae33c339aa138f705579752fee23e4f7b947863af89193b","source_snapshot_sha256":"28fd996b739a256dada034b9df49df7b85d2d3df0eb10a6fa055a41f5be25ce4"}},"jobs":[{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-regions-00"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":8,"shard_index":0},"depends_on":["09d","freeze","official-exposure","seen-rgb","setup-dino","setup-regions"],"id":"seen-regions-00","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-regions-01"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":8,"shard_index":1},"depends_on":["09d","freeze","official-exposure","seen-rgb","setup-dino","setup-regions"],"id":"seen-regions-01","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-regions-02"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":8,"shard_index":2},"depends_on":["09d","freeze","official-exposure","seen-rgb","setup-dino","setup-regions"],"id":"seen-regions-02","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-regions-03"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":8,"shard_index":3},"depends_on":["09d","freeze","official-exposure","seen-rgb","setup-dino","setup-regions"],"id":"seen-regions-03","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-regions-04"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":8,"shard_index":4},"depends_on":["09d","freeze","official-exposure","seen-rgb","setup-dino","setup-regions"],"id":"seen-regions-04","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-regions-05"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":8,"shard_index":5},"depends_on":["09d","freeze","official-exposure","seen-rgb","setup-dino","setup-regions"],"id":"seen-regions-05","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-regions-06"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":8,"shard_index":6},"depends_on":["09d","freeze","official-exposure","seen-rgb","setup-dino","setup-regions"],"id":"seen-regions-06","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-regions-07"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":8,"shard_index":7},"depends_on":["09d","freeze","official-exposure","seen-rgb","setup-dino","setup-regions"],"id":"seen-regions-07","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"region_merge","decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-regions"},"region_shards":[{"$resource":"seen-regions-00"},{"$resource":"seen-regions-01"},{"$resource":"seen-regions-02"},{"$resource":"seen-regions-03"},{"$resource":"seen-regions-04"},{"$resource":"seen-regions-05"},{"$resource":"seen-regions-06"},{"$resource":"seen-regions-07"}],"rgb_root":{"$resource":"seen-rgb"},"role":"seen"},"depends_on":["09d","freeze","official-exposure","seen-regions-00","seen-regions-01","seen-regions-02","seen-regions-03","seen-regions-04","seen-regions-05","seen-regions-06","seen-regions-07","seen-rgb","setup-cpu"],"id":"seen-regions","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-kaggle-preparation.ipynb","purpose":"region merge","runtime":"cpu"},{"config":{"action":"neutral","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"seen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"geometry_root":{"$resource":"seen-geometry"},"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-neutral"},"region_format":"current-native-regions","region_root":{"$resource":"seen-regions"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen"},"depends_on":["09d","freeze","official-exposure","seen-geometry","seen-regions","seen-rgb","setup-cpu"],"id":"seen-neutral","input_boundary":"trusted_preparation","notebook":"notebooks/input-preparation/current-kaggle-preparation.ipynb","purpose":"neutral","runtime":"cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"seen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"seen-cache-00-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"seen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-cache-00"},"region_format":"current-native-regions","region_root":{"$resource":"seen-regions"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","shard_count":8,"shard_index":0},"depends_on":["09d","freeze","official-exposure","seen-neutral","seen-regions","seen-rgb","setup-dino"],"id":"seen-cache-00","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"seen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"seen-cache-01-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"seen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-cache-01"},"region_format":"current-native-regions","region_root":{"$resource":"seen-regions"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","shard_count":8,"shard_index":1},"depends_on":["09d","freeze","official-exposure","seen-neutral","seen-regions","seen-rgb","setup-dino"],"id":"seen-cache-01","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"seen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"seen-cache-02-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"seen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-cache-02"},"region_format":"current-native-regions","region_root":{"$resource":"seen-regions"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","shard_count":8,"shard_index":2},"depends_on":["09d","freeze","official-exposure","seen-neutral","seen-regions","seen-rgb","setup-dino"],"id":"seen-cache-02","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"seen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"seen-cache-03-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"seen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-cache-03"},"region_format":"current-native-regions","region_root":{"$resource":"seen-regions"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","shard_count":8,"shard_index":3},"depends_on":["09d","freeze","official-exposure","seen-neutral","seen-regions","seen-rgb","setup-dino"],"id":"seen-cache-03","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"seen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"seen-cache-04-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"seen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-cache-04"},"region_format":"current-native-regions","region_root":{"$resource":"seen-regions"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","shard_count":8,"shard_index":4},"depends_on":["09d","freeze","official-exposure","seen-neutral","seen-regions","seen-rgb","setup-dino"],"id":"seen-cache-04","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"seen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"seen-cache-05-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"seen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-cache-05"},"region_format":"current-native-regions","region_root":{"$resource":"seen-regions"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","shard_count":8,"shard_index":5},"depends_on":["09d","freeze","official-exposure","seen-neutral","seen-regions","seen-rgb","setup-dino"],"id":"seen-cache-05","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"seen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"seen-cache-06-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"seen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-cache-06"},"region_format":"current-native-regions","region_root":{"$resource":"seen-regions"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","shard_count":8,"shard_index":6},"depends_on":["09d","freeze","official-exposure","seen-neutral","seen-regions","seen-rgb","setup-dino"],"id":"seen-cache-06","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"seen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"seen-cache-07-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"seen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-cache-07"},"region_format":"current-native-regions","region_root":{"$resource":"seen-regions"},"rgb_root":{"$resource":"seen-rgb"},"role":"seen","shard_count":8,"shard_index":7},"depends_on":["09d","freeze","official-exposure","seen-neutral","seen-regions","seen-rgb","setup-dino"],"id":"seen-cache-07","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"config":{"action":"cache_merge","cache_shards":[{"$resource":"seen-cache-00"},{"$resource":"seen-cache-01"},{"$resource":"seen-cache-02"},{"$resource":"seen-cache-03"},{"$resource":"seen-cache-04"},{"$resource":"seen-cache-05"},{"$resource":"seen-cache-06"},{"$resource":"seen-cache-07"}],"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"maximum_output_bytes":19327352832,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-cache"},"role":"seen"},"depends_on":["09d","freeze","official-exposure","seen-cache-00","seen-cache-01","seen-cache-02","seen-cache-03","seen-cache-04","seen-cache-05","seen-cache-06","seen-cache-07","setup-cpu"],"id":"seen-cache","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-kaggle-preparation.ipynb","purpose":"cache merge","runtime":"cpu"},{"config":{"action":"release","cache_root":{"$resource":"seen-cache"},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"10r-seen"},"role":"seen"},"depends_on":["09d","freeze","official-exposure","seen-cache","setup-cpu"],"id":"10r-seen","input_boundary":"model_only","notebook":"notebooks/official-evaluation/10r-toysad8k-official-release.ipynb","purpose":"release","runtime":"cpu"},{"config":{"action":"predict","cache_root":{"$resource":"10r-seen"},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"method_freeze_root":{"$resource":"freeze"},"mode":"automatic","official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"10a-automatic"},"role":"seen"},"depends_on":["09d","10r-seen","freeze","official-exposure","setup-cpu"],"id":"10a-automatic","input_boundary":"model_only","notebook":"notebooks/official-evaluation/10a-toysad8k-official-seen-frozen-inference.ipynb","purpose":"predict","runtime":"cpu"},{"config":{"action":"predict","cache_root":{"$resource":"10r-seen"},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"method_freeze_root":{"$resource":"freeze"},"mode":"assisted","official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"10a-assisted"},"role":"seen"},"depends_on":["09d","10r-seen","freeze","official-exposure","setup-cpu"],"id":"10a-assisted","input_boundary":"model_only","notebook":"notebooks/official-evaluation/10a-toysad8k-official-seen-frozen-inference.ipynb","purpose":"predict","runtime":"cpu"},{"config":{"action":"raw_truth","cache_root":{"$resource":"10r-seen"},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"geometry_root":{"$resource":"seen-geometry"},"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"seen-truth"},"prediction_roots":{"assisted":{"$resource":"10a-assisted"},"automatic":{"$resource":"10a-automatic"}},"raw_root":{"$resource":"raw"},"rgb_index_root":{"$resource":"seen-index"},"role":"seen"},"depends_on":["09d","10a-assisted","10a-automatic","10r-seen","freeze","official-exposure","seen-geometry","seen-index","setup-cpu"],"id":"seen-truth","input_boundary":"trusted_preparation","notebook":"notebooks/input-preparation/current-kaggle-preparation.ipynb","purpose":"raw truth","runtime":"cpu"},{"config":{"action":"rgb_release","decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"maximum_new_scenes":2437,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-rgb"},"raw_root":{"$resource":"raw"},"rgb_index_root":{"$resource":"unseen-index"},"role":"unseen"},"depends_on":["09d","freeze","official-exposure","setup-cpu","unseen-geometry","unseen-index"],"id":"unseen-rgb","input_boundary":"trusted_preparation","notebook":"notebooks/input-preparation/current-kaggle-preparation.ipynb","purpose":"rgb release","runtime":"cpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-00"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":0},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-00","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-01"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":1},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-01","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-02"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":2},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-02","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-03"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":3},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-03","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-04"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":4},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-04","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-05"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":5},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-05","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-06"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":6},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-06","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-07"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":7},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-07","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-08"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":8},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-08","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-09"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":9},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-09","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-10"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":10},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-10","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-11"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":11},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-11","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-12"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":12},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-12","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-13"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":13},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-13","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-14"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":14},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-14","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-15"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":15},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-15","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-16"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":16},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-16","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-17"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":17},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-17","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-18"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":18},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-18","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-19"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":19},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-19","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-20"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":20},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-20","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-21"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":21},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-21","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-22"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":22},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-22","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-23"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":23},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-23","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"extract","attempt_timeout_seconds":600,"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"locate_model_root":{"$resource":"models","path":"locateanything"},"maximum_new_views":2000,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions-24"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","sam_checkpoint":{"$resource":"models","path":"sam2/sam2.1_hiera_large.pt"},"shard_count":25,"shard_index":24},"depends_on":["09d","freeze","official-exposure","setup-dino","setup-regions","unseen-rgb"],"id":"unseen-regions-24","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-region-inference.ipynb","purpose":"Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase","runtime":"regions-gpu"},{"config":{"action":"region_merge","decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-regions"},"region_shards":[{"$resource":"unseen-regions-00"},{"$resource":"unseen-regions-01"},{"$resource":"unseen-regions-02"},{"$resource":"unseen-regions-03"},{"$resource":"unseen-regions-04"},{"$resource":"unseen-regions-05"},{"$resource":"unseen-regions-06"},{"$resource":"unseen-regions-07"},{"$resource":"unseen-regions-08"},{"$resource":"unseen-regions-09"},{"$resource":"unseen-regions-10"},{"$resource":"unseen-regions-11"},{"$resource":"unseen-regions-12"},{"$resource":"unseen-regions-13"},{"$resource":"unseen-regions-14"},{"$resource":"unseen-regions-15"},{"$resource":"unseen-regions-16"},{"$resource":"unseen-regions-17"},{"$resource":"unseen-regions-18"},{"$resource":"unseen-regions-19"},{"$resource":"unseen-regions-20"},{"$resource":"unseen-regions-21"},{"$resource":"unseen-regions-22"},{"$resource":"unseen-regions-23"},{"$resource":"unseen-regions-24"}],"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen"},"depends_on":["09d","freeze","official-exposure","setup-cpu","unseen-regions-00","unseen-regions-01","unseen-regions-02","unseen-regions-03","unseen-regions-04","unseen-regions-05","unseen-regions-06","unseen-regions-07","unseen-regions-08","unseen-regions-09","unseen-regions-10","unseen-regions-11","unseen-regions-12","unseen-regions-13","unseen-regions-14","unseen-regions-15","unseen-regions-16","unseen-regions-17","unseen-regions-18","unseen-regions-19","unseen-regions-20","unseen-regions-21","unseen-regions-22","unseen-regions-23","unseen-regions-24","unseen-rgb"],"id":"unseen-regions","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-kaggle-preparation.ipynb","purpose":"region merge","runtime":"cpu"},{"config":{"action":"neutral","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"geometry_root":{"$resource":"unseen-geometry"},"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-neutral"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen"},"depends_on":["09d","freeze","official-exposure","setup-cpu","unseen-geometry","unseen-regions","unseen-rgb"],"id":"unseen-neutral","input_boundary":"trusted_preparation","notebook":"notebooks/input-preparation/current-kaggle-preparation.ipynb","purpose":"neutral","runtime":"cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-00-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-00"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":0},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-00","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-01-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-01"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":1},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-01","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-02-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-02"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":2},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-02","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-03-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-03"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":3},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-03","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-04-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-04"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":4},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-04","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-05-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-05"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":5},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-05","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-06-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-06"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":6},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-06","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-07-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-07"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":7},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-07","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-08-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-08"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":8},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-08","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-09-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-09"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":9},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-09","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-10-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-10"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":10},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-10","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-11-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-11"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":11},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-11","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-12-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-12"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":12},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-12","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-13-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-13"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":13},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-13","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-14-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-14"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":14},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-14","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-15-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-15"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":15},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-15","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-16-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-16"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":16},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-16","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-17-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-17"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":17},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-17","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-18-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-18"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":18},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-18","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-19-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-19"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":19},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-19","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-20-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-20"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":20},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-20","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-21-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-21"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":21},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-21","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-22-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-22"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":22},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-22","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-23-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-23"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":23},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-23","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"compact_checkpoints":true,"config":{"action":"prepare_shard","decision_root":{"$resource":"09d"},"expected_region_lock_sha256":{"$resource":"unseen-regions","path":"artifact_lock.json","sha256":true},"exposure_root":{"$resource":"support","path":"original-exposure"},"feature_root":{"$output":"unseen-cache-24-features"},"freeze_root":{"$resource":"freeze"},"max_new_scenes":100,"max_new_units":2000,"maximum_output_bytes":16106127360,"method_freeze_root":{"$resource":"freeze"},"model_source":{"$resource":"models","path":"dino"},"neutral_root":{"$resource":"unseen-neutral"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache-24"},"region_format":"current-native-regions","region_root":{"$resource":"unseen-regions"},"rgb_root":{"$resource":"unseen-rgb"},"role":"unseen","shard_count":25,"shard_index":24},"depends_on":["09d","freeze","official-exposure","setup-dino","unseen-neutral","unseen-regions","unseen-rgb"],"id":"unseen-cache-24","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-native-regions-to-object-cache.ipynb","purpose":"Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes","runtime":"dino-cpu"},{"config":{"action":"cache_merge","cache_shards":[{"$resource":"unseen-cache-00"},{"$resource":"unseen-cache-01"},{"$resource":"unseen-cache-02"},{"$resource":"unseen-cache-03"},{"$resource":"unseen-cache-04"},{"$resource":"unseen-cache-05"},{"$resource":"unseen-cache-06"},{"$resource":"unseen-cache-07"},{"$resource":"unseen-cache-08"},{"$resource":"unseen-cache-09"},{"$resource":"unseen-cache-10"},{"$resource":"unseen-cache-11"},{"$resource":"unseen-cache-12"},{"$resource":"unseen-cache-13"},{"$resource":"unseen-cache-14"},{"$resource":"unseen-cache-15"},{"$resource":"unseen-cache-16"},{"$resource":"unseen-cache-17"},{"$resource":"unseen-cache-18"},{"$resource":"unseen-cache-19"},{"$resource":"unseen-cache-20"},{"$resource":"unseen-cache-21"},{"$resource":"unseen-cache-22"},{"$resource":"unseen-cache-23"},{"$resource":"unseen-cache-24"}],"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"maximum_output_bytes":19327352832,"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-cache"},"role":"unseen"},"depends_on":["09d","freeze","official-exposure","setup-cpu","unseen-cache-00","unseen-cache-01","unseen-cache-02","unseen-cache-03","unseen-cache-04","unseen-cache-05","unseen-cache-06","unseen-cache-07","unseen-cache-08","unseen-cache-09","unseen-cache-10","unseen-cache-11","unseen-cache-12","unseen-cache-13","unseen-cache-14","unseen-cache-15","unseen-cache-16","unseen-cache-17","unseen-cache-18","unseen-cache-19","unseen-cache-20","unseen-cache-21","unseen-cache-22","unseen-cache-23","unseen-cache-24"],"id":"unseen-cache","input_boundary":"model_only","notebook":"notebooks/input-preparation/current-kaggle-preparation.ipynb","purpose":"cache merge","runtime":"cpu"},{"config":{"action":"release","cache_root":{"$resource":"unseen-cache"},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"10r-unseen"},"role":"unseen"},"depends_on":["09d","freeze","official-exposure","setup-cpu","unseen-cache"],"id":"10r-unseen","input_boundary":"model_only","notebook":"notebooks/official-evaluation/10r-toysad8k-official-release.ipynb","purpose":"release","runtime":"cpu"},{"config":{"action":"predict","cache_root":{"$resource":"10r-unseen"},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"method_freeze_root":{"$resource":"freeze"},"mode":"automatic","official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"10a1-automatic"},"role":"unseen"},"depends_on":["09d","10r-unseen","freeze","official-exposure","setup-cpu"],"id":"10a1-automatic","input_boundary":"model_only","notebook":"notebooks/official-evaluation/10a1-toysad8k-official-unseen-frozen-inference.ipynb","purpose":"predict","runtime":"cpu"},{"config":{"action":"predict","cache_root":{"$resource":"10r-unseen"},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"method_freeze_root":{"$resource":"freeze"},"mode":"assisted","official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"10a1-assisted"},"role":"unseen"},"depends_on":["09d","10r-unseen","freeze","official-exposure","setup-cpu"],"id":"10a1-assisted","input_boundary":"model_only","notebook":"notebooks/official-evaluation/10a1-toysad8k-official-unseen-frozen-inference.ipynb","purpose":"predict","runtime":"cpu"},{"config":{"action":"raw_truth","cache_root":{"$resource":"10r-unseen"},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"geometry_root":{"$resource":"unseen-geometry"},"method_freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"unseen-truth"},"prediction_roots":{"assisted":{"$resource":"10a1-assisted"},"automatic":{"$resource":"10a1-automatic"}},"raw_root":{"$resource":"raw"},"rgb_index_root":{"$resource":"unseen-index"},"role":"unseen"},"depends_on":["09d","10a1-assisted","10a1-automatic","10r-unseen","freeze","official-exposure","setup-cpu","unseen-geometry","unseen-index"],"id":"unseen-truth","input_boundary":"trusted_preparation","notebook":"notebooks/input-preparation/current-kaggle-preparation.ipynb","purpose":"raw truth","runtime":"cpu"},{"config":{"action":"finalize","cache_roots":{"seen":{"$resource":"10r-seen"},"unseen":{"$resource":"10r-unseen"}},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"mode":"automatic","official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"10a-finalize-automatic"},"prediction_roots":{"seen":{"$resource":"10a-automatic"},"unseen":{"$resource":"10a1-automatic"}}},"depends_on":["09d","10a-automatic","10a1-automatic","10r-seen","10r-unseen","freeze","official-exposure","setup-cpu"],"id":"10a-finalize-automatic","input_boundary":"model_only","notebook":"notebooks/official-evaluation/10a-finalize-toysad8k-official-prediction-summary.ipynb","purpose":"finalize","runtime":"cpu"},{"config":{"action":"finalize","cache_roots":{"seen":{"$resource":"10r-seen"},"unseen":{"$resource":"10r-unseen"}},"decision_root":{"$resource":"09d"},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"mode":"assisted","official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"10a-finalize-assisted"},"prediction_roots":{"seen":{"$resource":"10a-assisted"},"unseen":{"$resource":"10a1-assisted"}}},"depends_on":["09d","10a-assisted","10a1-assisted","10r-seen","10r-unseen","freeze","official-exposure","setup-cpu"],"id":"10a-finalize-assisted","input_boundary":"model_only","notebook":"notebooks/official-evaluation/10a-finalize-toysad8k-official-prediction-summary.ipynb","purpose":"finalize","runtime":"cpu"},{"attach_resources":["runtime-tracking","wheels-tracking"],"config":{"action":"evaluate","cache_roots":{"seen":{"$resource":"10r-seen"},"unseen":{"$resource":"10r-unseen"}},"decision_root":{"$resource":"09d"},"evaluator_roots":{"seen":{"$resource":"seen-truth"},"unseen":{"$resource":"unseen-truth"}},"exposure_root":{"$resource":"support","path":"original-exposure"},"freeze_root":{"$resource":"freeze"},"official_exposure_root":{"$resource":"official-exposure"},"output_root":{"$output":"10b"},"prediction_roots":{"assisted":{"seen":{"$resource":"10a-assisted"},"unseen":{"$resource":"10a1-assisted"}},"automatic":{"seen":{"$resource":"10a-automatic"},"unseen":{"$resource":"10a1-automatic"}}},"prediction_summary_roots":{"assisted":{"$resource":"10a-finalize-assisted"},"automatic":{"$resource":"10a-finalize-automatic"}},"publish_tracking":true,"report_root":{"$output":"10b-report"}},"depends_on":["09d","10a-assisted","10a-automatic","10a-finalize-assisted","10a-finalize-automatic","10a1-assisted","10a1-automatic","10r-seen","10r-unseen","freeze","official-exposure","seen-truth","setup-cpu","setup-tracking","unseen-truth"],"id":"10b","input_boundary":"trusted_preparation","notebook":"notebooks/official-evaluation/10b-toysad8k-official-seen-unseen-evaluation.ipynb","purpose":"evaluate","runtime":"cpu"}],"methods":["winner_cluster_consensus","object_bank_top2"],"models_fixed":true,"official_roles":{"seen":715,"unseen":2437},"original_roles":{"calibration":600,"development":1000,"internal_lock":400},"queue_note":"Repeated cache jobs are100-scene resource shards of fixed methods, not new scoring experiments.","raw_dataset_reupload_required":false,"resources":{"08f1":{"availability":"existing_kaggle","external_source_job":"08f1","kind":"kernel","marker":"cohort_manifest.json","ref":"esreekarreddy/vpi-current-08f1","required_sha256":{"cohort_manifest.json":"01351302ce18183ad561e78e1464744d3ae96375e1080f9d5cffb0115b4d360e"},"suffixes":["08f1/model-cache"]},"09a":{"availability":"existing_kaggle","external_source_job":"09a","kind":"kernel","marker":"prediction_lock.json","ref":"esreekarreddy/vpi-current-09a","required_sha256":{"prediction_lock.json":"91d73ad63ae96f7219b35488b03e28af570b592fdbe3d70cfc8022982236968f"},"suffixes":["09a"]},"09b":{"availability":"existing_kaggle","contains_outcomes":true,"external_source_job":"09b","kind":"kernel","marker":"evaluation_lock.json","ref":"esreekarreddy/vpi-current-09b","required_sha256":{"evaluation_lock.json":"087188994d32f0ef8181a70cc5ebfbf2ddf8ebcbb34274eb57b66d7069a89526"},"suffixes":["09b"]},"09b1":{"availability":"existing_kaggle","external_source_job":"09b1","kind":"kernel","marker":"cohort_manifest.json","ref":"esreekarreddy/vpi-current-09b1","required_sha256":{"cohort_manifest.json":"a4b56aa0556ca0e30dfbdc4c89cc752677dac194e47abb0de52e1d439a039221"},"suffixes":["09b1/model-cache"]},"09c":{"availability":"existing_kaggle","external_source_job":"09c","kind":"kernel","marker":"prediction_lock.json","ref":"esreekarreddy/vpi-current-09c","required_sha256":{"prediction_lock.json":"c6c31649eda7dad81038d9e18cb0a2026738abf0794b79eb97a4127a0b0f0a0e"},"suffixes":["09c"]},"09d":{"availability":"existing_kaggle","contains_outcomes":true,"external_source_job":"09d","kind":"kernel","marker":"evaluation_lock.json","ref":"esreekarreddy/vpi-current-09d","required_sha256":{"evaluation_lock.json":"351bbaa12337ae120a425c72bcc534387a1f523e845cda40f7c9b956617aa780"},"suffixes":["09d"]},"10a-assisted":{"kind":"kernel","marker":"prediction_lock.json","producer_job":"10a-assisted","ref":"esreekarreddy/vpi-current-10a-assisted","suffixes":["10a-assisted"]},"10a-automatic":{"kind":"kernel","marker":"prediction_lock.json","producer_job":"10a-automatic","ref":"esreekarreddy/vpi-current-10a-automatic","suffixes":["10a-automatic"]},"10a-finalize-assisted":{"kind":"kernel","marker":"summary_lock.json","producer_job":"10a-finalize-assisted","ref":"esreekarreddy/vpi-current-10a-finalize-assisted","suffixes":["10a-finalize-assisted"]},"10a-finalize-automatic":{"kind":"kernel","marker":"summary_lock.json","producer_job":"10a-finalize-automatic","ref":"esreekarreddy/vpi-current-10a-finalize-automatic","suffixes":["10a-finalize-automatic"]},"10a1-assisted":{"kind":"kernel","marker":"prediction_lock.json","producer_job":"10a1-assisted","ref":"esreekarreddy/vpi-current-10a1-assisted","suffixes":["10a1-assisted"]},"10a1-automatic":{"kind":"kernel","marker":"prediction_lock.json","producer_job":"10a1-automatic","ref":"esreekarreddy/vpi-current-10a1-automatic","suffixes":["10a1-automatic"]},"10b":{"contains_outcomes":true,"kind":"kernel","marker":"evaluation_lock.json","producer_job":"10b","ref":"esreekarreddy/vpi-current-10b","suffixes":["10b"]},"10r-seen":{"kind":"kernel","marker":"cohort_manifest.json","producer_job":"10r-seen","ref":"esreekarreddy/vpi-current-10r-seen","suffixes":["10r-seen/model-input"]},"10r-unseen":{"kind":"kernel","marker":"cohort_manifest.json","producer_job":"10r-unseen","ref":"esreekarreddy/vpi-current-10r-unseen","suffixes":["10r-unseen/model-input"]},"administrative":{"availability":"existing_kaggle","kind":"dataset","marker":"restricted_source_map.jsonl","ref":"esreekarreddy/vpi-toysad-selection-private","suffixes":["toysad8k-train-2000-v2-administrative-vault"]},"calibration-cache":{"availability":"existing_kaggle","external_source_job":"calibration-cache","kind":"kernel","marker":"cohort_manifest.json","ref":"esreekarreddy/vpi-current-calibration-cache","required_sha256":{"cohort_manifest.json":"01351302ce18183ad561e78e1464744d3ae96375e1080f9d5cffb0115b4d360e"},"suffixes":["calibration-cache"]},"calibration-cache-00":{"availability":"existing_kaggle","external_source_job":"calibration-cache-00","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-cache-00","required_sha256":{"cache_shard_lock.json":"1a1e041982db1be4334371777737b3b675d8d6e3a6c48d1206b84839e76ec9cc"},"suffixes":["calibration-cache-00"]},"calibration-cache-01":{"availability":"existing_kaggle","external_source_job":"calibration-cache-01","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-cache-01","required_sha256":{"cache_shard_lock.json":"a16c4777509b2b317abcfa3064b2bd0fb30c7bcba7fd53055113b2b48ff68972"},"suffixes":["calibration-cache-01"]},"calibration-cache-02":{"availability":"existing_kaggle","external_source_job":"calibration-cache-02","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-cache-02","required_sha256":{"cache_shard_lock.json":"011238b5fde2410612bd377da8107500d93f64362b990faf10af907f0c7a5c6f"},"suffixes":["calibration-cache-02"]},"calibration-cache-03":{"availability":"existing_kaggle","external_source_job":"calibration-cache-03","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-cache-03","required_sha256":{"cache_shard_lock.json":"a5107cbacc600d9dc4b410c3fa8cdefa70e789c3907816a686c72adc643647c8"},"suffixes":["calibration-cache-03"]},"calibration-cache-04":{"availability":"existing_kaggle","external_source_job":"calibration-cache-04","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-cache-04","required_sha256":{"cache_shard_lock.json":"8c2824c9a719b5da17080900ba63d7ae3aaed8f88c35ab50307a6cca22994e2f"},"suffixes":["calibration-cache-04"]},"calibration-cache-05":{"availability":"existing_kaggle","external_source_job":"calibration-cache-05","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-cache-05","required_sha256":{"cache_shard_lock.json":"cc3e03f48dd4f5e164e967dde4b3e16ee524e83966893c675cc657918569de04"},"suffixes":["calibration-cache-05"]},"calibration-geometry":{"availability":"existing_kaggle","external_source_job":"calibration-geometry","kind":"kernel","marker":"geometry_manifest.json","ref":"esreekarreddy/vpi-current-calibration-geometry","required_sha256":{"geometry_manifest.json":"b87d30b20913c25deb0974feec743540a8927381aa4a82583baac3501bea86b3"},"suffixes":["calibration-geometry"]},"calibration-index":{"availability":"existing_kaggle","external_source_job":"calibration-index","kind":"kernel","marker":"rgb_index.json","ref":"esreekarreddy/vpi-current-calibration-index","required_sha256":{"rgb_index.json":"aa7e4a86350b100e0507eafc8aacec71286e7100117781f9a72ce9a3cf2fb5ea"},"suffixes":["calibration-index"]},"calibration-neutral":{"availability":"existing_kaggle","external_source_job":"calibration-neutral","kind":"kernel","marker":"neutral_correspondence.json","ref":"esreekarreddy/vpi-current-calibration-neutral","required_sha256":{"neutral_correspondence.json":"be807c28bef8d685ece1c226a5b93f8734ee51f3e6798c1f39c48b376dd4960b"},"suffixes":["calibration-neutral"]},"calibration-regions":{"availability":"existing_kaggle","external_source_job":"calibration-regions","kind":"kernel","marker":"artifact_lock.json","ref":"esreekarreddy/vpi-current-calibration-regions","required_sha256":{"artifact_lock.json":"0830ed1664dbaea8dbc47e60704c6df2ff02b81af49fc01c09de8b912f1a61c0"},"suffixes":["calibration-regions"]},"calibration-regions-00":{"availability":"existing_kaggle","external_source_job":"calibration-regions-00","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-regions-00","required_sha256":{"region_shard_lock.json":"9ba7ad6270222d09b749e9220b38cc7b12d9bee57a0b5161875840b978f3b6d7"},"suffixes":["calibration-regions-00"]},"calibration-regions-01":{"availability":"existing_kaggle","external_source_job":"calibration-regions-01","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-regions-01","required_sha256":{"region_shard_lock.json":"307354cc059c68f2762a00485cb7b8133b04cda1d08c9ed24a96a0c8571c9343"},"suffixes":["calibration-regions-01"]},"calibration-regions-02":{"availability":"existing_kaggle","external_source_job":"calibration-regions-02","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-regions-02","required_sha256":{"region_shard_lock.json":"b37f7763c0df892127e842531902b4899a5024f8c02457e1f94952c3b4cec302"},"suffixes":["calibration-regions-02"]},"calibration-regions-03":{"availability":"existing_kaggle","external_source_job":"calibration-regions-03","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-regions-03","required_sha256":{"region_shard_lock.json":"4c4644fef1eabeb6d24c62ae449eb22041b069e071ae1fed7b55224f45560766"},"suffixes":["calibration-regions-03"]},"calibration-regions-04":{"availability":"existing_kaggle","external_source_job":"calibration-regions-04","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-regions-04","required_sha256":{"region_shard_lock.json":"33979168a4e8f6c9322518dbbb55f8e52243fa7f8466bc06104446ad714c8337"},"suffixes":["calibration-regions-04"]},"calibration-regions-05":{"availability":"existing_kaggle","external_source_job":"calibration-regions-05","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-calibration-regions-05","required_sha256":{"region_shard_lock.json":"af6c2c218a6d84fe7983aa2957979f5d4689d7c7f7e16dd1aaf8d61e181e6454"},"suffixes":["calibration-regions-05"]},"calibration-rgb":{"availability":"existing_kaggle","external_source_job":"calibration-rgb","kind":"kernel","marker":"package_record.json","ref":"esreekarreddy/vpi-current-calibration-rgb","required_sha256":{"package_record.json":"542ade928cbc87b0ecfd902616c39af96c58748b527064f3d592fa9691a2bb12"},"suffixes":["calibration-rgb/model-input"]},"calibration-truth":{"availability":"existing_kaggle","contains_outcomes":true,"external_source_job":"calibration-truth","kind":"kernel","marker":"current_evaluator_manifest.json","ref":"esreekarreddy/vpi-current-calibration-truth","required_sha256":{"current_evaluator_manifest.json":"c3a18e4c4691359fbb7946ab01dda333f5d49a3dcf84903dd5e11c2135e9b97a"},"suffixes":["calibration-truth"]},"development-cache":{"availability":"existing_kaggle","external_source_job":"development-cache","kind":"kernel","marker":"cohort_manifest.json","ref":"esreekarreddy/vpi-current-development-cache","required_sha256":{"cohort_manifest.json":"b8548176bc1cf0608b59157fb1eb2e00ce2acfdea50d9ef6bab288e624d04816"},"suffixes":["development-cache"]},"development-cache-00":{"availability":"existing_kaggle","external_source_job":"development-cache-00","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-00","required_sha256":{"cache_shard_lock.json":"391764e98ed6bd6ef1d9b8d0ef38692fc4f5cce8057bc10e38b92c9f870f93ba"},"suffixes":["development-cache-00"]},"development-cache-01":{"availability":"existing_kaggle","external_source_job":"development-cache-01","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-01","required_sha256":{"cache_shard_lock.json":"023091ed2dd8fdc7107e92a675fd802438d41ea7ee5cb2594a25de6b510220d0"},"suffixes":["development-cache-01"]},"development-cache-02":{"availability":"existing_kaggle","external_source_job":"development-cache-02","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-02","required_sha256":{"cache_shard_lock.json":"8ccd0b88d09a6afc0c11d6a29f8983d62c59e719a10b920bdd2b1412f39441ce"},"suffixes":["development-cache-02"]},"development-cache-03":{"availability":"existing_kaggle","external_source_job":"development-cache-03","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-03","required_sha256":{"cache_shard_lock.json":"60ddd249f26849dee72d67a6c93ebbc2af0ea82d21156501c9f035698d7e9c16"},"suffixes":["development-cache-03"]},"development-cache-04":{"availability":"existing_kaggle","external_source_job":"development-cache-04","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-04","required_sha256":{"cache_shard_lock.json":"2005451cee34715d3cf50fa41c4ddf08b2de0e6fe4206c261b94b8bab20283e0"},"suffixes":["development-cache-04"]},"development-cache-05":{"availability":"existing_kaggle","external_source_job":"development-cache-05","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-05","required_sha256":{"cache_shard_lock.json":"1cb58e46ef3afef0fda3476719089cee30235dde0bb403fc822ef4dcd57ce40b"},"suffixes":["development-cache-05"]},"development-cache-06":{"availability":"existing_kaggle","external_source_job":"development-cache-06","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-06","required_sha256":{"cache_shard_lock.json":"931162fbfc5d020089ba7eaa5cd0f4b47e6ebe7579b56e15e52c492b80af1af0"},"suffixes":["development-cache-06"]},"development-cache-07":{"availability":"existing_kaggle","external_source_job":"development-cache-07","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-07","required_sha256":{"cache_shard_lock.json":"32728878cae2b833753fbdf624900d5c7e8fba6b47cb85d58670e82a9403d757"},"suffixes":["development-cache-07"]},"development-cache-08":{"availability":"existing_kaggle","external_source_job":"development-cache-08","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-08","required_sha256":{"cache_shard_lock.json":"78c29d6d344ee0809915ddbf1f6bf2a94984d3965ac4c4b57012ca36c6d68dc6"},"suffixes":["development-cache-08"]},"development-cache-09":{"availability":"existing_kaggle","external_source_job":"development-cache-09","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-development-cache-09","required_sha256":{"cache_shard_lock.json":"87a43a15a5d37e353826d9896c5a4fac49c6554ad03e055e658cc5b317808973"},"suffixes":["development-cache-09"]},"development-evaluate":{"availability":"existing_kaggle","contains_outcomes":true,"external_source_job":"development-evaluate","kind":"kernel","marker":"evaluation_lock.json","ref":"esreekarreddy/vpi-current-development-evaluate","required_sha256":{"evaluation_lock.json":"e6c80e78c86957d0ba1dfb486dc0ad251178f728233003c2cf270eec90f31652"},"suffixes":["development-evaluate"]},"development-geometry":{"availability":"existing_kaggle","external_source_job":"development-geometry","kind":"kernel","marker":"geometry_manifest.json","ref":"esreekarreddy/vpi-current-development-geometry","required_sha256":{"geometry_manifest.json":"c9b8abb9b4f6593e6435e19ba00983c327f86e50098dc5b475b61829f329e895"},"suffixes":["development-geometry"]},"development-index":{"availability":"existing_kaggle","external_source_job":"development-index","kind":"kernel","marker":"rgb_index.json","ref":"esreekarreddy/vpi-current-development-index","required_sha256":{"rgb_index.json":"a813ed0247bf035533348930c450c850e274da96c4d3506da875dfbdee506736"},"suffixes":["development-index"]},"development-neutral":{"availability":"existing_kaggle","external_source_job":"development-neutral","kind":"kernel","marker":"neutral_correspondence.json","ref":"esreekarreddy/vpi-current-development-neutral","required_sha256":{"neutral_correspondence.json":"8d8bf36f6e8163778a8203850e12e973f9f2aeba48d9c5e660aaec7aea7beb9d"},"suffixes":["development-neutral"]},"development-predict":{"availability":"existing_kaggle","external_source_job":"development-predict","kind":"kernel","marker":"prediction_lock.json","ref":"esreekarreddy/vpi-current-development-predict","required_sha256":{"prediction_lock.json":"e372c4c80bf485f2d132f73db584fb8afd028859038b3e7b7c950c8969d1ba9d"},"suffixes":["development-predict"]},"development-regions":{"availability":"existing_kaggle","kind":"kernel","marker":"full-development-lock.json","ref":"esreekarreddy/toysad-8k-frozen-development-pipeline","required_sha256":{"full-development-lock.json":"9d8505d2bccdcf9d6e59ebf39c52676278517ff6564c391aa88cb4fb113c050c"},"suffixes":["toysad8k_frozen_development_pipeline"]},"development-truth":{"availability":"existing_kaggle","contains_outcomes":true,"external_source_job":"development-truth","kind":"kernel","marker":"current_evaluator_manifest.json","ref":"esreekarreddy/vpi-current-development-truth","required_sha256":{"current_evaluator_manifest.json":"7e3dfe798160ad6d571573792fc19b8657bdd307e257a2d74d4b3e862ec0924f"},"suffixes":["development-truth"]},"freeze":{"availability":"existing_kaggle","external_source_job":"freeze","kind":"kernel","marker":"method_freeze.json","ref":"esreekarreddy/vpi-current-freeze","required_sha256":{"method_freeze.json":"57b8a32532c249f462f77240e277e9c35a206b7d18c34f63a410a5d280f71b50"},"suffixes":["freeze"]},"internal_lock-cache":{"availability":"existing_kaggle","external_source_job":"internal_lock-cache","kind":"kernel","marker":"cohort_manifest.json","ref":"esreekarreddy/vpi-current-internal-lock-cache","required_sha256":{"cohort_manifest.json":"a4b56aa0556ca0e30dfbdc4c89cc752677dac194e47abb0de52e1d439a039221"},"suffixes":["internal_lock-cache"]},"internal_lock-cache-00":{"availability":"existing_kaggle","external_source_job":"internal_lock-cache-00","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-internal-lock-cache-00","required_sha256":{"cache_shard_lock.json":"0234bfe6fc453be91ac0208c5795093244c765ff056935e65f3aa816a486b65f"},"suffixes":["internal_lock-cache-00"]},"internal_lock-cache-01":{"availability":"existing_kaggle","external_source_job":"internal_lock-cache-01","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-internal-lock-cache-01","required_sha256":{"cache_shard_lock.json":"e851d23b547cc55801e4702844141ccf393c3da2f2bb70a93282746f528098cd"},"suffixes":["internal_lock-cache-01"]},"internal_lock-cache-02":{"availability":"existing_kaggle","external_source_job":"internal_lock-cache-02","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-internal-lock-cache-02","required_sha256":{"cache_shard_lock.json":"86a3382d1f36f340485a6c2d06f1f48e29cece7390b737769b31c48de800616f"},"suffixes":["internal_lock-cache-02"]},"internal_lock-cache-03":{"availability":"existing_kaggle","external_source_job":"internal_lock-cache-03","kind":"kernel","marker":"cache_shard_lock.json","ref":"esreekarreddy/vpi-current-internal-lock-cache-03","required_sha256":{"cache_shard_lock.json":"e8bf050e46a73d5a95a54e4870bd19cee175b6e472b55c3e6037544facaf9482"},"suffixes":["internal_lock-cache-03"]},"internal_lock-geometry":{"availability":"existing_kaggle","external_source_job":"internal_lock-geometry","kind":"kernel","marker":"geometry_manifest.json","ref":"esreekarreddy/vpi-current-internal-lock-geometry","required_sha256":{"geometry_manifest.json":"d7d59f794aafa26b9d8e2f0e2935ad1129887010ec1ee9c718cbe347425a4fb5"},"suffixes":["internal_lock-geometry"]},"internal_lock-index":{"availability":"existing_kaggle","external_source_job":"internal_lock-index","kind":"kernel","marker":"rgb_index.json","ref":"esreekarreddy/vpi-current-internal-lock-index","required_sha256":{"rgb_index.json":"901263a0104f4d753a78e3a2d4cacdcc6015bf3a63fadb2ff5aeeb227e203726"},"suffixes":["internal_lock-index"]},"internal_lock-neutral":{"availability":"existing_kaggle","external_source_job":"internal_lock-neutral","kind":"kernel","marker":"neutral_correspondence.json","ref":"esreekarreddy/vpi-current-internal-lock-neutral","required_sha256":{"neutral_correspondence.json":"4fa4aec3afa6e146bcf85ee6010a16a9ca85f7598dc94d5571b78583da85512b"},"suffixes":["internal_lock-neutral"]},"internal_lock-regions":{"availability":"existing_kaggle","external_source_job":"internal_lock-regions","kind":"kernel","marker":"artifact_lock.json","ref":"esreekarreddy/vpi-current-internal-lock-regions","required_sha256":{"artifact_lock.json":"289bd402472e56e7417c57b8f0084c7feee78378b8b4123a915ecd06c7a32da5"},"suffixes":["internal_lock-regions"]},"internal_lock-regions-00":{"availability":"existing_kaggle","external_source_job":"internal_lock-regions-00","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-internal-lock-regions-00","required_sha256":{"region_shard_lock.json":"30e338fe7e91d73fd72deeae909b0be6e6a908eb0015da52f807b283f8d95e76"},"suffixes":["internal_lock-regions-00"]},"internal_lock-regions-01":{"availability":"existing_kaggle","external_source_job":"internal_lock-regions-01","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-internal-lock-regions-01","required_sha256":{"region_shard_lock.json":"3f1d06f7f75c9fe41df64b82af650b42ccbbbc61d5e2f0c4f815f007de318d2a"},"suffixes":["internal_lock-regions-01"]},"internal_lock-regions-02":{"availability":"existing_kaggle","external_source_job":"internal_lock-regions-02","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-internal-lock-regions-02","required_sha256":{"region_shard_lock.json":"5691d1e90d8ea8d0726cb4a456bf417d1ebbf86d1600f26b611564cf4aea5b51"},"suffixes":["internal_lock-regions-02"]},"internal_lock-regions-03":{"availability":"existing_kaggle","external_source_job":"internal_lock-regions-03","kind":"kernel","marker":"region_shard_lock.json","ref":"esreekarreddy/vpi-current-internal-lock-regions-03","required_sha256":{"region_shard_lock.json":"a441e15b4f19db7fa48eb2d11543ee075e1a2c28370b27558696c79f6261b7af"},"suffixes":["internal_lock-regions-03"]},"internal_lock-rgb":{"availability":"existing_kaggle","external_source_job":"internal_lock-rgb","kind":"kernel","marker":"package_record.json","ref":"esreekarreddy/vpi-current-internal-lock-rgb","required_sha256":{"package_record.json":"07063d5ce596a035f5c693f35b4d7e6136ac041265c587bcc6f94775d7e26094"},"suffixes":["internal_lock-rgb/model-input"]},"internal_lock-truth":{"availability":"existing_kaggle","contains_outcomes":true,"external_source_job":"internal_lock-truth","kind":"kernel","marker":"current_evaluator_manifest.json","ref":"esreekarreddy/vpi-current-internal-lock-truth","required_sha256":{"current_evaluator_manifest.json":"907f3a899a5c59a3c98d4e3a0474c49bb15dcff3c0bec3d1af2684ed162753db"},"suffixes":["internal_lock-truth"]},"models":{"availability":"existing_kaggle","external_source_job":"setup-dino","kind":"kernel","marker":"model_assets_receipt.json","ref":"esreekarreddy/vpi-current-setup-dino","required_sha256":{"model_assets_receipt.json":"f407fdc4c45291526d82bf39fa0f5a841a6dbd301a6d6febace9d3eaa2f79698"},"suffixes":["setup-dino/model-assets"]},"official-exposure":{"availability":"existing_kaggle","external_source_job":"official-exposure","kind":"kernel","marker":"official_exposure_ledger.json","ref":"esreekarreddy/vpi-current-official-exposure","required_sha256":{"official_exposure_ledger.json":"041a3c3d2b77bf2503172ac3436f36b05a707265218129136bb7f937760ed8fe"},"suffixes":["official-exposure"]},"raw":{"availability":"existing_kaggle","contains_outcomes":true,"kind":"dataset","marker":"package-checksums.sha256","ref":"esreekarreddy/toysad8k-full-private-eda","suffixes":["","toysad8k-full-private-eda"]},"reuse100":{"availability":"prepared_local_upload","kind":"dataset","marker":"feature_shard_lock.json","ref":"esreekarreddy/vpi-current-development100-feature-reuse","suffixes":[""]},"rgb-confirm":{"availability":"existing_kaggle","kind":"dataset","marker":"package_record.json","ref":"esreekarreddy/vpi-toysad-ds2-confirmation-rgb","suffixes":["model-input",""]},"rgb-remainder":{"availability":"existing_kaggle","kind":"dataset","marker":"package_record.json","ref":"esreekarreddy/vpi-toysad-development-remainder-model-input","suffixes":["development-remainder/model-input","model-input",""]},"rgb-screen":{"availability":"existing_kaggle","kind":"dataset","marker":"package_record.json","ref":"esreekarreddy/vpi-toysad-ds1-screen100-model-input","suffixes":["model-input",""]},"runtime-cpu":{"availability":"existing_kaggle","external_source_job":"setup-cpu","kind":"kernel","marker":"runtime_profile_lock.json","ref":"esreekarreddy/vpi-current-setup-cpu","required_sha256":{"runtime_profile_lock.json":"246cbde33f8c880f6a26bb23dad896b44ec9b39234e7dcd439df5c22c4aebd5d"},"suffixes":["setup-cpu/runtime-profile"]},"runtime-dino":{"availability":"existing_kaggle","external_source_job":"setup-dino","kind":"kernel","marker":"runtime_profile_lock.json","ref":"esreekarreddy/vpi-current-setup-dino","required_sha256":{"runtime_profile_lock.json":"4462eef3a141c0d647b0d418c91e98e134522240f678027d7bef8e0aebc15a0f"},"suffixes":["setup-dino/runtime-profile"]},"runtime-regions":{"availability":"existing_kaggle","external_source_job":"setup-regions","kind":"kernel","marker":"runtime_profile_lock.json","ref":"esreekarreddy/vpi-current-setup-regions","required_sha256":{"runtime_profile_lock.json":"f92df2f676fda1c06794c22ea04e996ce298958f7af082ae64c250052c4a4d2c"},"suffixes":["setup-regions/runtime-profile"]},"runtime-tracking":{"availability":"existing_kaggle","external_source_job":"setup-tracking","kind":"kernel","marker":"runtime_profile_lock.json","ref":"esreekarreddy/vpi-current-setup-tracking","required_sha256":{"runtime_profile_lock.json":"fba340a3438b742d110fd053cd5748ef2d6715d965ae70b50330b600b901192c"},"suffixes":["setup-tracking/runtime-profile"]},"seen-cache":{"kind":"kernel","marker":"cohort_manifest.json","producer_job":"seen-cache","ref":"esreekarreddy/vpi-current-seen-cache","suffixes":["seen-cache"]},"seen-cache-00":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"seen-cache-00","ref":"esreekarreddy/vpi-current-seen-cache-00","suffixes":["seen-cache-00"]},"seen-cache-01":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"seen-cache-01","ref":"esreekarreddy/vpi-current-seen-cache-01","suffixes":["seen-cache-01"]},"seen-cache-02":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"seen-cache-02","ref":"esreekarreddy/vpi-current-seen-cache-02","suffixes":["seen-cache-02"]},"seen-cache-03":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"seen-cache-03","ref":"esreekarreddy/vpi-current-seen-cache-03","suffixes":["seen-cache-03"]},"seen-cache-04":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"seen-cache-04","ref":"esreekarreddy/vpi-current-seen-cache-04","suffixes":["seen-cache-04"]},"seen-cache-05":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"seen-cache-05","ref":"esreekarreddy/vpi-current-seen-cache-05","suffixes":["seen-cache-05"]},"seen-cache-06":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"seen-cache-06","ref":"esreekarreddy/vpi-current-seen-cache-06","suffixes":["seen-cache-06"]},"seen-cache-07":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"seen-cache-07","ref":"esreekarreddy/vpi-current-seen-cache-07","suffixes":["seen-cache-07"]},"seen-geometry":{"availability":"existing_kaggle","external_source_job":"seen-geometry","kind":"kernel","marker":"geometry_manifest.json","ref":"esreekarreddy/vpi-current-seen-geometry","required_sha256":{"geometry_manifest.json":"52ab97d718ce199bfb73503cb17a7dd503d1bbd8228587bfac5a90589294492e"},"suffixes":["seen-geometry"]},"seen-index":{"availability":"existing_kaggle","external_source_job":"seen-index","kind":"kernel","marker":"rgb_index.json","ref":"esreekarreddy/vpi-current-seen-index","required_sha256":{"rgb_index.json":"30e8a04d606bf0f423635fff06ace30e2f1e8b345b4300908af7326e4a371506"},"suffixes":["seen-index"]},"seen-neutral":{"kind":"kernel","marker":"neutral_correspondence.json","producer_job":"seen-neutral","ref":"esreekarreddy/vpi-current-seen-neutral","suffixes":["seen-neutral"]},"seen-regions":{"kind":"kernel","marker":"artifact_lock.json","producer_job":"seen-regions","ref":"esreekarreddy/vpi-current-seen-regions","suffixes":["seen-regions"]},"seen-regions-00":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"seen-regions-00","ref":"esreekarreddy/vpi-current-seen-regions-00","suffixes":["seen-regions-00"]},"seen-regions-01":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"seen-regions-01","ref":"esreekarreddy/vpi-current-seen-regions-01","suffixes":["seen-regions-01"]},"seen-regions-02":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"seen-regions-02","ref":"esreekarreddy/vpi-current-seen-regions-02","suffixes":["seen-regions-02"]},"seen-regions-03":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"seen-regions-03","ref":"esreekarreddy/vpi-current-seen-regions-03","suffixes":["seen-regions-03"]},"seen-regions-04":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"seen-regions-04","ref":"esreekarreddy/vpi-current-seen-regions-04","suffixes":["seen-regions-04"]},"seen-regions-05":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"seen-regions-05","ref":"esreekarreddy/vpi-current-seen-regions-05","suffixes":["seen-regions-05"]},"seen-regions-06":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"seen-regions-06","ref":"esreekarreddy/vpi-current-seen-regions-06","suffixes":["seen-regions-06"]},"seen-regions-07":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"seen-regions-07","ref":"esreekarreddy/vpi-current-seen-regions-07","suffixes":["seen-regions-07"]},"seen-rgb":{"availability":"existing_kaggle","external_source_job":"seen-rgb","kind":"kernel","marker":"package_record.json","ref":"esreekarreddy/vpi-current-seen-rgb","required_sha256":{"package_record.json":"2d641015f5c612d585b17514ed2106cb48bc4fd2d2b2916bb26ff101dba8ab53"},"suffixes":["seen-rgb/model-input"]},"seen-truth":{"contains_outcomes":true,"kind":"kernel","marker":"current_evaluator_manifest.json","producer_job":"seen-truth","ref":"esreekarreddy/vpi-current-seen-truth","suffixes":["seen-truth"]},"selection":{"availability":"existing_kaggle","kind":"dataset","marker":"experiment_roles.jsonl","ref":"esreekarreddy/vpi-toysad-selection-private","suffixes":["toysad8k-train-2000-v2"]},"source":{"availability":"prepare_upload","kind":"dataset","marker":"prepared_source_manifest.json","ref":"esreekarreddy/vpi-current-evaluation-source","suffixes":["source",""]},"support":{"availability":"prepare_upload","kind":"dataset","marker":"support_manifest.json","ref":"esreekarreddy/vpi-current-evaluation-support","suffixes":[""]},"unseen-cache":{"kind":"kernel","marker":"cohort_manifest.json","producer_job":"unseen-cache","ref":"esreekarreddy/vpi-current-unseen-cache","suffixes":["unseen-cache"]},"unseen-cache-00":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-00","ref":"esreekarreddy/vpi-current-unseen-cache-00","suffixes":["unseen-cache-00"]},"unseen-cache-01":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-01","ref":"esreekarreddy/vpi-current-unseen-cache-01","suffixes":["unseen-cache-01"]},"unseen-cache-02":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-02","ref":"esreekarreddy/vpi-current-unseen-cache-02","suffixes":["unseen-cache-02"]},"unseen-cache-03":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-03","ref":"esreekarreddy/vpi-current-unseen-cache-03","suffixes":["unseen-cache-03"]},"unseen-cache-04":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-04","ref":"esreekarreddy/vpi-current-unseen-cache-04","suffixes":["unseen-cache-04"]},"unseen-cache-05":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-05","ref":"esreekarreddy/vpi-current-unseen-cache-05","suffixes":["unseen-cache-05"]},"unseen-cache-06":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-06","ref":"esreekarreddy/vpi-current-unseen-cache-06","suffixes":["unseen-cache-06"]},"unseen-cache-07":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-07","ref":"esreekarreddy/vpi-current-unseen-cache-07","suffixes":["unseen-cache-07"]},"unseen-cache-08":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-08","ref":"esreekarreddy/vpi-current-unseen-cache-08","suffixes":["unseen-cache-08"]},"unseen-cache-09":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-09","ref":"esreekarreddy/vpi-current-unseen-cache-09","suffixes":["unseen-cache-09"]},"unseen-cache-10":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-10","ref":"esreekarreddy/vpi-current-unseen-cache-10","suffixes":["unseen-cache-10"]},"unseen-cache-11":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-11","ref":"esreekarreddy/vpi-current-unseen-cache-11","suffixes":["unseen-cache-11"]},"unseen-cache-12":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-12","ref":"esreekarreddy/vpi-current-unseen-cache-12","suffixes":["unseen-cache-12"]},"unseen-cache-13":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-13","ref":"esreekarreddy/vpi-current-unseen-cache-13","suffixes":["unseen-cache-13"]},"unseen-cache-14":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-14","ref":"esreekarreddy/vpi-current-unseen-cache-14","suffixes":["unseen-cache-14"]},"unseen-cache-15":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-15","ref":"esreekarreddy/vpi-current-unseen-cache-15","suffixes":["unseen-cache-15"]},"unseen-cache-16":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-16","ref":"esreekarreddy/vpi-current-unseen-cache-16","suffixes":["unseen-cache-16"]},"unseen-cache-17":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-17","ref":"esreekarreddy/vpi-current-unseen-cache-17","suffixes":["unseen-cache-17"]},"unseen-cache-18":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-18","ref":"esreekarreddy/vpi-current-unseen-cache-18","suffixes":["unseen-cache-18"]},"unseen-cache-19":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-19","ref":"esreekarreddy/vpi-current-unseen-cache-19","suffixes":["unseen-cache-19"]},"unseen-cache-20":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-20","ref":"esreekarreddy/vpi-current-unseen-cache-20","suffixes":["unseen-cache-20"]},"unseen-cache-21":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-21","ref":"esreekarreddy/vpi-current-unseen-cache-21","suffixes":["unseen-cache-21"]},"unseen-cache-22":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-22","ref":"esreekarreddy/vpi-current-unseen-cache-22","suffixes":["unseen-cache-22"]},"unseen-cache-23":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-23","ref":"esreekarreddy/vpi-current-unseen-cache-23","suffixes":["unseen-cache-23"]},"unseen-cache-24":{"kind":"kernel","marker":"cache_shard_lock.json","producer_job":"unseen-cache-24","ref":"esreekarreddy/vpi-current-unseen-cache-24","suffixes":["unseen-cache-24"]},"unseen-geometry":{"availability":"existing_kaggle","external_source_job":"unseen-geometry","kind":"kernel","marker":"geometry_manifest.json","ref":"esreekarreddy/vpi-current-unseen-geometry","required_sha256":{"geometry_manifest.json":"4815f8dabb475574d85bc587571013e48592f4f0d96ffbc911ac8d95ca277932"},"suffixes":["unseen-geometry"]},"unseen-index":{"availability":"existing_kaggle","external_source_job":"unseen-index","kind":"kernel","marker":"rgb_index.json","ref":"esreekarreddy/vpi-current-unseen-index","required_sha256":{"rgb_index.json":"37ea69cd586e234b05e6d5d9cb0ad6941013e6b1a972de70df1e81ecfd15a392"},"suffixes":["unseen-index"]},"unseen-neutral":{"kind":"kernel","marker":"neutral_correspondence.json","producer_job":"unseen-neutral","ref":"esreekarreddy/vpi-current-unseen-neutral","suffixes":["unseen-neutral"]},"unseen-regions":{"kind":"kernel","marker":"artifact_lock.json","producer_job":"unseen-regions","ref":"esreekarreddy/vpi-current-unseen-regions","suffixes":["unseen-regions"]},"unseen-regions-00":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-00","ref":"esreekarreddy/vpi-current-unseen-regions-00","suffixes":["unseen-regions-00"]},"unseen-regions-01":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-01","ref":"esreekarreddy/vpi-current-unseen-regions-01","suffixes":["unseen-regions-01"]},"unseen-regions-02":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-02","ref":"esreekarreddy/vpi-current-unseen-regions-02","suffixes":["unseen-regions-02"]},"unseen-regions-03":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-03","ref":"esreekarreddy/vpi-current-unseen-regions-03","suffixes":["unseen-regions-03"]},"unseen-regions-04":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-04","ref":"esreekarreddy/vpi-current-unseen-regions-04","suffixes":["unseen-regions-04"]},"unseen-regions-05":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-05","ref":"esreekarreddy/vpi-current-unseen-regions-05","suffixes":["unseen-regions-05"]},"unseen-regions-06":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-06","ref":"esreekarreddy/vpi-current-unseen-regions-06","suffixes":["unseen-regions-06"]},"unseen-regions-07":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-07","ref":"esreekarreddy/vpi-current-unseen-regions-07","suffixes":["unseen-regions-07"]},"unseen-regions-08":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-08","ref":"esreekarreddy/vpi-current-unseen-regions-08","suffixes":["unseen-regions-08"]},"unseen-regions-09":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-09","ref":"esreekarreddy/vpi-current-unseen-regions-09","suffixes":["unseen-regions-09"]},"unseen-regions-10":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-10","ref":"esreekarreddy/vpi-current-unseen-regions-10","suffixes":["unseen-regions-10"]},"unseen-regions-11":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-11","ref":"esreekarreddy/vpi-current-unseen-regions-11","suffixes":["unseen-regions-11"]},"unseen-regions-12":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-12","ref":"esreekarreddy/vpi-current-unseen-regions-12","suffixes":["unseen-regions-12"]},"unseen-regions-13":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-13","ref":"esreekarreddy/vpi-current-unseen-regions-13","suffixes":["unseen-regions-13"]},"unseen-regions-14":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-14","ref":"esreekarreddy/vpi-current-unseen-regions-14","suffixes":["unseen-regions-14"]},"unseen-regions-15":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-15","ref":"esreekarreddy/vpi-current-unseen-regions-15","suffixes":["unseen-regions-15"]},"unseen-regions-16":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-16","ref":"esreekarreddy/vpi-current-unseen-regions-16","suffixes":["unseen-regions-16"]},"unseen-regions-17":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-17","ref":"esreekarreddy/vpi-current-unseen-regions-17","suffixes":["unseen-regions-17"]},"unseen-regions-18":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-18","ref":"esreekarreddy/vpi-current-unseen-regions-18","suffixes":["unseen-regions-18"]},"unseen-regions-19":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-19","ref":"esreekarreddy/vpi-current-unseen-regions-19","suffixes":["unseen-regions-19"]},"unseen-regions-20":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-20","ref":"esreekarreddy/vpi-current-unseen-regions-20","suffixes":["unseen-regions-20"]},"unseen-regions-21":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-21","ref":"esreekarreddy/vpi-current-unseen-regions-21","suffixes":["unseen-regions-21"]},"unseen-regions-22":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-22","ref":"esreekarreddy/vpi-current-unseen-regions-22","suffixes":["unseen-regions-22"]},"unseen-regions-23":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-23","ref":"esreekarreddy/vpi-current-unseen-regions-23","suffixes":["unseen-regions-23"]},"unseen-regions-24":{"kind":"kernel","marker":"region_shard_lock.json","producer_job":"unseen-regions-24","ref":"esreekarreddy/vpi-current-unseen-regions-24","suffixes":["unseen-regions-24"]},"unseen-rgb":{"kind":"kernel","marker":"package_record.json","producer_job":"unseen-rgb","ref":"esreekarreddy/vpi-current-unseen-rgb","suffixes":["unseen-rgb/model-input"]},"unseen-truth":{"contains_outcomes":true,"kind":"kernel","marker":"current_evaluator_manifest.json","producer_job":"unseen-truth","ref":"esreekarreddy/vpi-current-unseen-truth","suffixes":["unseen-truth"]},"wheels-cpu":{"availability":"existing_kaggle","external_source_job":"setup-cpu","kind":"kernel","marker":"wheelhouse_manifest.json","ref":"esreekarreddy/vpi-current-setup-cpu","required_sha256":{"wheelhouse_manifest.json":"ab746146158ca180f5838e28f62dc19fb291df241dd13b1e12ee558c739e3332"},"suffixes":["setup-cpu/wheelhouse"]},"wheels-dino":{"availability":"existing_kaggle","external_source_job":"setup-dino","kind":"kernel","marker":"wheelhouse_manifest.json","ref":"esreekarreddy/vpi-current-setup-dino","required_sha256":{"wheelhouse_manifest.json":"908cd70d0da8788f6cfd2e752d7dc1ddef11c5d81349653eaa08546b3e014207"},"suffixes":["setup-dino/wheelhouse"]},"wheels-regions":{"availability":"existing_kaggle","external_source_job":"setup-regions","kind":"kernel","marker":"wheelhouse_manifest.json","ref":"esreekarreddy/vpi-current-setup-regions","required_sha256":{"wheelhouse_manifest.json":"b95309ec18a1c4de59cdb09fa1d7817668db2d7a1432fe8254f86a378da0893f"},"suffixes":["setup-regions/wheelhouse"]},"wheels-tracking":{"availability":"existing_kaggle","external_source_job":"setup-tracking","kind":"kernel","marker":"wheelhouse_manifest.json","ref":"esreekarreddy/vpi-current-setup-tracking","required_sha256":{"wheelhouse_manifest.json":"00d27db6c0b0a4e826e43d8a353274c81b062d6bbefaa3f5d0d2d6b0a42d12ae"},"suffixes":["setup-tracking/wheelhouse"]}},"runtime_readiness":{"sam2_code":{"distribution":"SAM-2","preflight_marker":"sam2_preflight_verified","repository":"https://github.com/facebookresearch/sam2","resource":"wheels-regions","revision":"2b90b9f5ceec907a1c18123530e92e794ad901a4"}},"schedule_count":19,"schema":"vpi-current-kaggle-launch-plan/v1","scientific_runs_started":true}')
    _JOB_ID = 'seen-neutral'
    configure_runtime(LAUNCH_PLAN, _JOB_ID, CODE_ROOT)
    LAUNCH_BINDING = resolve_job(LAUNCH_PLAN, _JOB_ID)
    import subprocess
    import uuid
    _profile_relative = "requirements/current-evaluation-runtime.json"
    _attempt = uuid.uuid4().hex[:12]
    _target = _runtime.runtime_target("/tmp/vpi-runtime-" + _JOB_ID + "-" + _attempt)
    os.environ["VPI_PACKAGE_TARGET"] = str(_target)
    _isolated_profile = Path("/kaggle/working/runtime-profiles") / (_JOB_ID + "-" + _attempt)
    _prepare_command = [sys.executable, str(CODE_ROOT / "scripts/prepare_runtime_environment.py"),
                        "--profile", str(CODE_ROOT / _profile_relative),
                        "--wheelhouse", os.environ["VPI_WHEELHOUSE"],
                        "--target", str(_target), "--output-root", str(_isolated_profile)]
    subprocess.run(_prepare_command, check=True, timeout=1020)
    _base_profile = json.loads((CODE_ROOT / _profile_relative).read_text())
    _base_contract = _base_profile.get("runtime_contract", _base_profile)
    _runtime.observed_contract(_isolated_profile, CODE_ROOT / _profile_relative, _base_contract)
    _dependency_receipt = json.loads((_target / "dependency_receipt.json").read_text())
    if _dependency_receipt.get("runtime_contract") != _base_contract:
        raise ValueError("Fresh isolated dependency target differs from the registered profile")
    _plan_path = Path("/tmp") / ("vpi-current-plan-" + _JOB_ID + "-" + _attempt + ".json")
    _plan_path.write_text(json.dumps(LAUNCH_PLAN, sort_keys=True, separators=(",", ":")))
    _child_path = Path("/tmp") / ("vpi-current-cpu-" + _JOB_ID + "-" + _attempt + ".py")
    _child_path.write_text('from pathlib import Path\nimport importlib\nimport json\nimport os\nimport sys\nimport types\n\nCODE_ROOT = Path(os.environ["VPI_RESOLVED_CODE_ROOT"]).resolve(strict=True)\nNOTEBOOK_RELATIVE = \'notebooks/input-preparation/current-kaggle-preparation.ipynb\'\nRUNTIME_PROFILE = None\nsys.path.insert(0, str(CODE_ROOT / "scripts"))\nsys.path.insert(0, str(CODE_ROOT / "src"))\nimport current_notebook_runtime as _runtime\nBOOTSTRAP = _runtime.bootstrap(CODE_ROOT, NOTEBOOK_RELATIVE, runtime_profile=RUNTIME_PROFILE)\nCODE_PROVENANCE = BOOTSTRAP["provenance"]\nROOT = CODE_ROOT\n\ndef display(value):\n    print(json.dumps(value, sort_keys=True, default=str), flush=True)\n\nclass Image:\n    def __init__(self, filename):\n        self.filename = str(filename)\n    def __str__(self):\n        return self.filename\n\n# Canonical report cells use IPython only to construct display objects.  Keep\n# the child dependency target closed by supplying that tiny display surface.\n_ipython = types.ModuleType("IPython")\n_ipython.__path__ = []\n_ipython.get_ipython = lambda: None\n_ipython.version_info = (0, 0)\n_ipython_display = types.ModuleType("IPython.display")\n_ipython_display.display = display\n_ipython_display.Image = Image\n_ipython.display = _ipython_display\nsys.modules.setdefault("IPython", _ipython)\nsys.modules.setdefault("IPython.display", _ipython_display)\n\nLAUNCH_PLAN = json.loads(Path(os.environ["VPI_LAUNCH_PLAN_PATH"]).read_text())\n_JOB_ID = \'seen-neutral\'\n_PHASE_PATH = Path("/kaggle/working/runtime-receipts") / (_JOB_ID + "-isolated-child.json")\n_PHASE = {"schema": "vpi-isolated-cpu-child/v1", "job_id": _JOB_ID,\n          "status": "bootstrap_verified", "scientific_run_started": False}\n_runtime.atomic_json(_PHASE_PATH, _PHASE)\ntry:\n    _dependency_target = Path(os.environ["VPI_PACKAGE_TARGET"]).resolve(strict=True)\n    _ambient_packages = [\n        str(Path(item).resolve()) for item in sys.path\n        if item and ("site-packages" in item or "dist-packages" in item)\n        and not Path(item).resolve().is_relative_to(_dependency_target)\n    ]\n    if _ambient_packages:\n        raise RuntimeError("Isolated CPU child can see ambient site packages")\n    import numpy\n    import pandas as pd\n    import scipy\n    import matplotlib\n    import PIL\n    for _module in (numpy, pd, scipy, matplotlib, PIL):\n        if not Path(_module.__file__).resolve().is_relative_to(_dependency_target):\n            raise RuntimeError("Pinned CPU dependency resolved outside the private target")\n    _PHASE.update(status="runtime_imports_verified", numpy=str(numpy.__version__),\n                  pandas=str(pd.__version__), scipy=str(scipy.__version__))\n    _runtime.atomic_json(_PHASE_PATH, _PHASE)\n    _PHASE.update(status="science_running", scientific_run_started=True)\n    _runtime.atomic_json(_PHASE_PATH, _PHASE)\n    CONFIG = _runtime.resolve_config(\'preparation\', ROOT)\n    CONFIG[\'output_root\'] = str(_runtime.writable_output(CONFIG[\'output_root\']))\n    from current_stage_execution import run_stage\n    ACTION = CONFIG[\'action\']\n\n    def prepare():\n        if ACTION == \'freeze\':\n            from vpi_next_evaluation.contracts import prepare_method_freeze\n            return prepare_method_freeze(CONFIG[\'reference_root\'], CONFIG[\'output_root\'], ROOT)\n        if ACTION in {\'rgb_index\',\'rgb_release\'}:\n            from vpi_next_evaluation.native_rgb import prepare_rgb_index, release_rgb\n            return (prepare_rgb_index if ACTION==\'rgb_index\' else release_rgb)(CONFIG,ROOT)\n        if ACTION == \'raw_geometry\':\n            from current_official_geometry import prepare_official_mask_geometry\n            return prepare_official_mask_geometry(CONFIG)\n        if ACTION == \'raw_truth\':\n            from current_truth_conversion import convert_raw_evaluator_with_typed_rejections\n            return convert_raw_evaluator_with_typed_rejections(CONFIG)\n        if ACTION == \'neutral\':\n            from vpi_next_evaluation.native_evaluator import prepare_neutral\n            return prepare_neutral(CONFIG)\n        if ACTION == \'cache_merge\':\n            from vpi_next_evaluation.cache_shards import merge_cache_shards\n            return merge_cache_shards(CONFIG,ROOT)\n        if ACTION == \'region_merge\':\n            from vpi_next_evaluation.region_shards import merge_region_shards\n            return merge_region_shards(CONFIG,ROOT)\n        if ACTION == \'prepare_shard\':\n            from vpi_next_evaluation.native_cache import prepare_shard\n            return prepare_shard(CONFIG,ROOT)\n        if ACTION == \'official_exposure\':\n            from current_official_exposure import prepare_official_exposure_from_geometry\n            return prepare_official_exposure_from_geometry(CONFIG)\n        raise ValueError(\'Unknown prepared launch operation\')\n\n    RESULT = run_stage(\'preparation-\'+ACTION,CONFIG,prepare,CODE_PROVENANCE)\n    display(RESULT)\n    _PHASE.update(status="science_complete")\n    _runtime.atomic_json(_PHASE_PATH, _PHASE)\n    from current_launch import complete_job\n    JOB_COMPLETION = complete_job(\n        LAUNCH_PLAN, _JOB_ID, CODE_PROVENANCE["snapshot_sha256"])\n    print(JOB_COMPLETION, flush=True)\n    _PHASE.update(status="complete", tracking_status=\'not_applicable\')\n    _runtime.atomic_json(_PHASE_PATH, _PHASE)\nexcept BaseException as _child_error:\n    _PHASE.update(status="failed", error_type=type(_child_error).__name__,\n                  scientific_results_preserved=True)\n    _runtime.atomic_json(_PHASE_PATH, _PHASE)\n    raise\n')
    _environment = dict(os.environ)
    _environment["VPI_RESOLVED_CODE_ROOT"] = str(CODE_ROOT)
    _environment["VPI_LAUNCH_PLAN_PATH"] = str(_plan_path)
    _environment["PYTHONUNBUFFERED"] = "1"
    _environment["PYTHONDONTWRITEBYTECODE"] = "1"
    _environment["MPLBACKEND"] = "Agg"
    _environment["MPLCONFIGDIR"] = "/tmp/vpi-matplotlib-" + _attempt
    _environment["PYTHONPATH"] = os.pathsep.join(
        [str(_target), str(CODE_ROOT / "scripts"), str(CODE_ROOT / "src")])
    _child_invoked = True
    _completed = subprocess.run(
        [sys.executable, "-S", "-B", str(_child_path)], env=_environment,
        check=True, timeout=14400)
    _completion_path = Path("/kaggle/working/job_completion.json")
    _completion = json.loads(_completion_path.read_text())
    if (_completion.get("schema") != "vpi-current-job-completion/v1"
            or _completion.get("job_id") != _JOB_ID
            or _completion.get("status") != "outputs_complete"):
        raise ValueError("Isolated CPU child did not write this job's completion receipt")
    print(json.dumps({"launch_binding": LAUNCH_BINDING,
                      "child_returncode": _completed.returncode,
                      "completion": _completion}, sort_keys=True, default=str))
except BaseException as _bootstrap_error:
    _failure_root = Path('/kaggle/working/runtime-receipts') if Path('/kaggle').is_dir() else Path.cwd()/'.local/runtime-receipts'
    _failure_root.mkdir(parents=True, exist_ok=True)
    _phase_path = _failure_root / ('seen-neutral' + '-isolated-child.json')
    try:
        _phase = json.loads(_phase_path.read_text()) if _phase_path.is_file() else {}
    except Exception:
        _phase = {}
    _child_invoked_value = bool(locals().get('_child_invoked', False))
    _failure = {'status': 'isolated_child_failed' if _child_invoked_value else 'bootstrap_failed',
                'error_type': _phase.get('error_type', type(_bootstrap_error).__name__),
                'scientific_run_started': _phase.get('scientific_run_started', False),
                'scientific_results_preserved': _phase.get('scientific_results_preserved', _child_invoked_value)}
    _temporary = _failure_root/'bootstrap-failure.json.partial'
    _temporary.write_text(json.dumps(_failure, sort_keys=True)+'\n')
    os.replace(_temporary, _failure_root/'bootstrap-failure.json')
    raise


{
  "status": "runtime_profile_prepared",
  "python": "3.12.13",
  "profile_sha256": "b43d05c002fcc11da3e0a724e35082c61483927932941824a202afe3b3f8279d",
  "private_target_prepared": true,
  "models_loaded": false,
  "datasets_opened": false,
  "scientific_methods_executed": false
}
{"anomaly_labels_opened": false, "observation_count": 53294, "status": "complete", "unmatched_count": 2777}
{'schema': 'vpi-current-job-completion/v1', 'status': 'outputs_complete', 'job_id': 'seen-neutral', 'plan_sha256': '0661bc93e66f803dd8a6a36c4207a4cef2b22df65ea7f185db8dcfc82a4c81d5', 'source_snapshot_sha256': 'd2c37bb6d8824d2e085c6206b70e5ddfb9927eef25336f58bcd41f33bc1a7296', 'outputs': {'seen-neutral': '5b96d01dafc8d7bced75675fc5a7a6ab6a26ffe2029c4f1fac7841aa5a0f29c5'}, 'full_download_verified': False}
{"child_returncode": 0, "completion": {"full_download_verified": false, "job_id": "seen-neutral", "outputs": {"seen-neutral": "5b96d01dafc8d7bced75675fc5a7a6ab6a26ffe2029c4f1fac7841aa5a0f29c5"}, "plan_s